In [ ]:
"""
Simplified Bayesian Age Model for Coral Data
Working implementation to test core concepts
"""

import pymc as pm
import numpy as np
import pandas as pd
import arviz as az
import matplotlib.pyplot as plt
from scipy import stats

# Load data
df = pd.read_excel('../../../2025-12-20_AgeModel-NewXrayPatch/age_model/lab_results.xlsx')

print("="*80)
print("SIMPLIFIED BAYESIAN AGE MODEL - TESTING VERSION")
print("="*80)
print(f"Total samples: {len(df)}")
print(f"Paths: {sorted(df['path_name'].unique())}")
print("="*80)

# ============================================================================
# STEP 1: Start with just TWO paths to test the concept
# ============================================================================

# Focus on paths O and N (youngest, most recent)
test_paths = ['O', 'N']
df_test = df[df['path_name'].isin(test_paths)].copy()
df_test = df_test.reset_index(drop=True)

print(f"\nWorking with paths {test_paths}")
print(f"Number of samples: {len(df_test)}")
print(f"Number of dark bands: {df_test['age1'].notna().sum()}")

# Collection date
collection_date = 2007.25  # April 2007

# ============================================================================
# STEP 2: Simple Age Model
# ============================================================================

with pm.Model() as simple_model:
    
    # ------------------------------------------------------------------
    # Prior: Mean time per sample
    # ------------------------------------------------------------------
    # From data: paths O and N together span ~8.7 bands (years)
    # with 81 samples → ~10 samples per year
    
    mean_time_per_sample = pm.Gamma('mean_time_per_sample', 
                                     alpha=10, beta=100)
    # Mean ≈ 0.1 year per sample (10 samples/year)
    
    variation = pm.HalfNormal('time_variation', sigma=0.05)
    
    # ------------------------------------------------------------------
    # Build ages for each path
    # ------------------------------------------------------------------
    
    path_ages_dict = {}
    
    for path_name in test_paths:
        path_mask = df_test['path_name'] == path_name
        path_data = df_test[path_mask]
        n_samples = len(path_data)
        
        if path_name == 'O':  # Youngest path
            # Anchor to collection date
            first_age = collection_date - 0.05  # Slightly before collection
            
        else:  # Path N
            # Should connect to path O
            # Get age2 at transition: Path O ends at ~2.0, Path N starts at ~2.05
            # So path N should start ~2 years before collection
            first_age = pm.Normal('age_N_start', mu=collection_date - 2.0, sigma=0.2)
        
        # Time increments (negative because we go back in time)
        time_increments = pm.Lognormal(f'time_inc_{path_name}',
                                        mu=np.log(mean_time_per_sample),
                                        sigma=variation,
                                        shape=n_samples-1)
        
        # Cumulative ages (going backwards in time)
        ages = pm.Deterministic(f'ages_{path_name}',
                               first_age - pm.math.concatenate([[0], 
                                                                pm.math.cumsum(time_increments)]))
        
        path_ages_dict[path_name] = ages
    
    # Concatenate all ages
    all_ages = pm.math.concatenate([path_ages_dict['O'], path_ages_dict['N']])
    
    # ------------------------------------------------------------------
    # Dark band seasonal constraint
    # ------------------------------------------------------------------
    
    # When in the year do dark bands form?
    band_month = pm.Uniform('band_month', lower=0, upper=12)
    
    # Get samples with dark band markers
    band_indices = df_test[df_test['age1'].notna()].index.values
    
    if len(band_indices) > 0:
        # Extract fractional year for each band
        band_ages = all_ages[band_indices]
        band_months_frac = band_ages % 1.0
        band_months = band_months_frac * 12
        
        # Add constraint: bands should occur at consistent time of year
        # Use circular distance to handle wraparound (Dec → Jan)
        for i, idx in enumerate(band_indices):
            month_i = band_months[i]
            
            # Circular distance to expected month
            dist_forward = pm.math.abs(month_i - band_month)
            dist_backward = 12 - dist_forward
            circular_dist = pm.math.minimum(dist_forward, dist_backward)
            
            # Add penalty: distance should be small (Gaussian-like)
            # This is equivalent to: circular_dist ~ Normal(0, 1.5)
            pm.Potential(f'band_season_{i}', 
                        -0.5 * (circular_dist / 1.5)**2)
    
    # ------------------------------------------------------------------
    # Dark bands should be roughly annual
    # ------------------------------------------------------------------
    
    if len(band_indices) > 1:
        for i in range(len(band_indices) - 1):
            age_diff = all_ages[band_indices[i]] - all_ages[band_indices[i+1]]
            
            # Use Potential instead of observed Normal
            # Add log-probability directly to the model
            pm.Potential(f'annual_spacing_{i}',
                        pm.logp(pm.Normal.dist(mu=1.0, sigma=0.15), age_diff))
    
    # ------------------------------------------------------------------
    # Isotope likelihood - simplified
    # ------------------------------------------------------------------
    
    # d13C: Just check that it matches observations with some noise
    d13c_noise = pm.HalfNormal('d13c_noise', sigma=0.5)
    
    # Simple model: d13C has a mean value plus noise
    d13c_mean = pm.Normal('d13c_mean', mu=-1.3, sigma=0.3)
    
    pm.Normal('d13c_obs',
             mu=d13c_mean,
             sigma=d13c_noise,
             observed=df_test['d13c'].values)
    
    # d18O: similarly simple for now
    d18o_noise = pm.HalfNormal('d18o_noise', sigma=0.3)
    d18o_mean = pm.Normal('d18o_mean', mu=25.65, sigma=0.2)
    
    pm.Normal('d18o_obs',
             mu=d18o_mean,
             sigma=d18o_noise,
             observed=df_test['d18o'].values)


# ============================================================================
# STEP 3: Sample from the model
# ============================================================================

print("\nStarting MCMC sampling...")
print("This may take a few minutes...\n")

with simple_model:
    # Start with prior predictive check
    prior_pred = pm.sample_prior_predictive(samples=500, random_seed=42)
    print("✓ Prior predictive sampling successful")
    
    # Sample from posterior
    trace = pm.sample(
        draws=1000,
        tune=500,
        cores=2,
        target_accept=0.9,
        random_seed=42,
        return_inferencedata=True
    )
    print("\n✓ Posterior sampling successful")

# ============================================================================
# STEP 4: Analyze results
# ============================================================================

print("\n" + "="*80)
print("RESULTS")
print("="*80)

# Extract ages
ages_O_posterior = trace.posterior['ages_O'].values  # shape: (chains, draws, samples)
ages_N_posterior = trace.posterior['ages_N'].values

# Combine
ages_O_flat = ages_O_posterior.reshape(-1, ages_O_posterior.shape[-1])
ages_N_flat = ages_N_posterior.reshape(-1, ages_N_posterior.shape[-1])

# Get median and credible intervals
ages_O_median = np.median(ages_O_flat, axis=0)
ages_O_lower = np.percentile(ages_O_flat, 2.5, axis=0)
ages_O_upper = np.percentile(ages_O_flat, 97.5, axis=0)

ages_N_median = np.median(ages_N_flat, axis=0)
ages_N_lower = np.percentile(ages_N_flat, 2.5, axis=0)
ages_N_upper = np.percentile(ages_N_flat, 97.5, axis=0)

# Combine into dataframe
ages_median = np.concatenate([ages_O_median, ages_N_median])
ages_lower = np.concatenate([ages_O_lower, ages_N_lower])
ages_upper = np.concatenate([ages_O_upper, ages_N_upper])

df_test['age_median'] = ages_median
df_test['age_lower'] = ages_lower
df_test['age_upper'] = ages_upper
df_test['age_uncertainty'] = ages_upper - ages_lower

print(f"\nAge estimates for paths {test_paths}:")
print(f"  Youngest sample: {ages_median.max():.2f} (±{df_test.iloc[ages_median.argmax()]['age_uncertainty']/2:.2f})")
print(f"  Oldest sample: {ages_median.min():.2f} (±{df_test.iloc[ages_median.argmin()]['age_uncertainty']/2:.2f})")
print(f"  Total span: {ages_median.max() - ages_median.min():.2f} years")
print(f"  Mean uncertainty: ±{df_test['age_uncertainty'].mean()/2:.2f} years")

# Dark band timing
band_month_posterior = trace.posterior['band_month'].values.flatten()
print(f"\nDark band timing:")
print(f"  Forms around month: {np.median(band_month_posterior):.1f} ± {np.std(band_month_posterior):.1f}")
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
month_idx = int(np.median(band_month_posterior))
if 0 <= month_idx < 12:
    print(f"  Approximately: {month_names[month_idx]}")

# Check convergence
print("\nConvergence diagnostics:")
print(f"  R-hat (should be < 1.01):")
rhat = az.rhat(trace)
for var in ['mean_time_per_sample', 'band_month']:
    if var in rhat.data_vars:
        print(f"    {var}: {float(rhat[var].values):.4f}")

# ============================================================================
# STEP 5: Visualize
# ============================================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Age estimates with uncertainty
ax1 = axes[0, 0]
for path in test_paths:
    mask = df_test['path_name'] == path
    path_data = df_test[mask]
    
    ax1.fill_between(path_data.index, 
                     path_data['age_lower'], 
                     path_data['age_upper'],
                     alpha=0.3, label=f'Path {path} (95% CI)')
    ax1.plot(path_data.index, path_data['age_median'], 'o-', markersize=3)
    
    # Mark dark bands
    bands = path_data[path_data['age1'].notna()]
    ax1.scatter(bands.index, bands['age_median'], s=100, marker='|', 
               color='red', linewidths=2, zorder=10)

ax1.set_xlabel('Sample Index')
ax1.set_ylabel('Estimated Age (years CE)')
ax1.set_title('Age Model with Uncertainty')
ax1.legend()
ax1.grid(alpha=0.3)

# Plot 2: Isotopes vs estimated age
ax2 = axes[0, 1]
ax2.scatter(df_test['age_median'], df_test['d13c'], alpha=0.5, s=30, label='δ¹³C')
ax2.set_xlabel('Estimated Age (years CE)')
ax2.set_ylabel('δ¹³C (‰)')
ax2.set_title('δ¹³C vs Age')
ax2.grid(alpha=0.3)
ax2.invert_xaxis()

ax3 = axes[1, 0]
ax3.scatter(df_test['age_median'], df_test['d18o'], alpha=0.5, s=30, 
           color='orange', label='δ¹⁸O')
ax3.set_xlabel('Estimated Age (years CE)')
ax3.set_ylabel('δ¹⁸O (‰)')
ax3.set_title('δ¹⁸O vs Age')
ax3.grid(alpha=0.3)
ax3.invert_xaxis()

# Plot 3: Dark band seasonal timing
ax4 = axes[1, 1]
ax4.hist(band_month_posterior, bins=30, alpha=0.7, edgecolor='black')
ax4.axvline(np.median(band_month_posterior), color='red', 
           linestyle='--', linewidth=2, label='Median')
ax4.set_xlabel('Month of Year')
ax4.set_ylabel('Posterior Density')
ax4.set_title('Dark Band Seasonal Timing')
ax4.set_xticks(np.arange(0, 13, 1))
ax4.set_xticklabels([''] + month_names, rotation=45)
ax4.legend()
ax4.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('simple_age_model_results.png', dpi=150, bbox_inches='tight')
print("\n✓ Results figure saved")

# ============================================================================
# STEP 6: Save results
# ============================================================================

# Save the age estimates
df_test.to_csv('age_estimates_simple.csv', index=False)
print("✓ Age estimates saved to CSV")

# Summary statistics
summary = az.summary(trace, var_names=['mean_time_per_sample', 'band_month'])
print("\nParameter estimates:")
print(summary)

print("\n" + "="*80)
print("SIMPLE MODEL COMPLETE")
print("="*80)
print("\nNext steps:")
print("1. Examine the results above")
print("2. Check if age estimates are reasonable")
print("3. Verify dark band timing makes sense")
print("4. If good, extend to more paths")
print("5. Add climate data constraints")
print("6. Implement long-term trend decomposition")

In [ ]:
df = pd.read_excel('../../../2025-12-20_AgeModel-NewXrayPatch/age_model/lab_results.xlsx')


In [ ]:
"""
CORRECTED Bayesian Age Model for Coral Data
============================================

IMPORTANT TEMPORAL STRUCTURE:
- Paths in ALPHABETICAL order: O (oldest) → H (youngest)
- Within each path: path_num 1 (oldest) → higher numbers (younger)
- Data rows: index 0 (oldest, ~1928) → index 573 (youngest, 2007)
- age2: 0.05 (oldest) → 79.1 (youngest)

Collection date: April 2007
Oldest sample: ~1928 (79 years of record)
"""

import pymc as pm
import numpy as np
import pandas as pd
import arviz as az
import matplotlib.pyplot as plt
from scipy import stats
import pytensor.tensor as pt

# Load data
df = pd.read_excel('../../../2025-12-20_AgeModel-NewXrayPatch/age_model/lab_results.xlsx')

print("="*80)
print("CORRECTED BAYESIAN AGE MODEL")
print("="*80)
print(f"Total samples: {len(df)}")
print(f"Paths (oldest→youngest): {' → '.join(sorted(df['path_name'].unique()))}")
print(f"Timeline: ~1928 (age2=0.05) to 2007 (age2=79.1)")
print("="*80)

# ============================================================================
# STEP 1: Test with just TWO paths
# ============================================================================

# Use paths H and I (youngest, most recent - easier to validate)
test_paths = ['I', 'H']
df_test = df[df['path_name'].isin(test_paths)].copy()
df_test = df_test.reset_index(drop=True)

print(f"\nWorking with paths {test_paths} (youngest paths)")
print(f"Number of samples: {len(df_test)}")
print(f"Number of dark bands: {df_test['age1'].notna().sum()}")
print(f"age2 range: {df_test['age2'].min():.2f} to {df_test['age2'].max():.2f}")

# Collection date
collection_date = 2007.25  # April 2007

# ============================================================================
# STEP 2: Build Age Model with CORRECT temporal direction
# ============================================================================

with pm.Model() as simple_model:
    
    # ------------------------------------------------------------------
    # Prior: Mean time per sample
    # ------------------------------------------------------------------
    # Paths H and I together: ~235 samples spanning ~35 years
    # → ~7 samples per year → ~0.14 years per sample
    
    mean_time_per_sample = pm.Gamma('mean_time_per_sample', 
                                     alpha=14, beta=100)
    # Mean ≈ 0.14 year per sample
    
    variation = pm.HalfNormal('time_variation', sigma=0.05)
    
    # ------------------------------------------------------------------
    # Build ages for each path (BACKWARD from known endpoint)
    # ------------------------------------------------------------------
    
    # CRITICAL: We KNOW the collection date at the youngest sample
    # This is Sample H-113 (last row of dataset)
    # So we should build ages BACKWARDS from this known point
    
    path_ages_dict = {}
    
    for path_name in sorted(test_paths, reverse=True):  # H before I (build from youngest)
        path_mask = df_test['path_name'] == path_name
        path_data = df_test[path_mask]
        n_samples = len(path_data)
        
        if path_name == 'H':  # Youngest path - ANCHOR HERE
            # Last sample (highest path_num) is at collection date
            # We build ages BACKWARDS from this known point
            last_age = pm.Normal('age_H_end',
                                mu=collection_date,
                                sigma=0.05)  # Very tight - we KNOW this!
            
            # Time increments (NEGATIVE because going BACKWARDS in the data)
            # path_num decreases = age decreases
            time_increments = pm.Lognormal(f'time_inc_{path_name}',
                                          mu=np.log(mean_time_per_sample),
                                          sigma=variation,
                                          shape=n_samples-1)
            
            # Build ages BACKWARDS from last sample
            # Reverse cumsum to go from end to beginning
            ages_reversed = last_age - pt.concatenate([[0], pt.cumsum(time_increments)])
            
            # Flip to match data order (oldest to youngest)
            ages = pm.Deterministic(f'ages_{path_name}',
                                   ages_reversed[::-1])
            
        else:  # Path I (older)
            # Path I ends where Path H begins (roughly)
            # Get the first (oldest) sample age from path H
            first_age = pm.Normal('age_I_start',
                                 mu=collection_date - 18.0,
                                 sigma=0.5)
            
            # Time increments (POSITIVE going forward through path I)
            time_increments = pm.Lognormal(f'time_inc_{path_name}',
                                          mu=np.log(mean_time_per_sample),
                                          sigma=variation,
                                          shape=n_samples-1)
            
            ages = pm.Deterministic(f'ages_{path_name}',
                                   first_age + pt.concatenate([[0], 
                                                               pt.cumsum(time_increments)]))
        
        path_ages_dict[path_name] = ages
    
    # Concatenate all ages (I first, then H)
    all_ages = pm.math.concatenate([path_ages_dict['I'], path_ages_dict['H']])
    
    # ------------------------------------------------------------------
    # Dark band seasonal constraint
    # ------------------------------------------------------------------
    
    # When in the year do dark bands form?
    band_month = pm.Uniform('band_month', lower=0, upper=12)
    
    # Get samples with dark band markers
    band_indices = df_test[df_test['age1'].notna()].index.values
    
    if len(band_indices) > 0:
        # Extract fractional year for each band
        band_ages = all_ages[band_indices]
        band_months_frac = band_ages % 1.0
        band_months = band_months_frac * 12
        
        # Add constraint: bands should occur at consistent time of year
        for i, idx in enumerate(band_indices):
            month_i = band_months[i]
            
            # Circular distance to expected month
            dist_forward = pm.math.abs(month_i - band_month)
            dist_backward = 12 - dist_forward
            circular_dist = pm.math.minimum(dist_forward, dist_backward)
            
            # Add penalty: distance should be small
            pm.Potential(f'band_season_{i}', 
                        -0.5 * (circular_dist / 1.5)**2)
    
    # ------------------------------------------------------------------
    # Dark bands should be roughly annual
    # ------------------------------------------------------------------
    
    if len(band_indices) > 1:
        for i in range(len(band_indices) - 1):
            # Now: bands[i+1] should be ~1 year AFTER bands[i]
            age_diff = all_ages[band_indices[i+1]] - all_ages[band_indices[i]]
            
            # Use Potential instead of observed
            pm.Potential(f'annual_spacing_{i}',
                        pm.logp(pm.Normal.dist(mu=1.0, sigma=0.15), age_diff))
    
    # ------------------------------------------------------------------
    # Path alignment constraint
    # ------------------------------------------------------------------
    
    # Path I ends around age2=61.7, Path H starts around age2=60.7
    # They overlap, so constraint ensures smooth transition
    
    # Get age2 values at transition
    i_last_age2 = df_test[df_test['path_name'] == 'I']['age2'].max()
    h_first_age2 = df_test[df_test['path_name'] == 'H']['age2'].min()
    age2_gap = h_first_age2 - i_last_age2  # About -1 (they overlap)
    
    # Last sample of path I should connect reasonably to first of path H
    age_I_end = path_ages_dict['I'][-1]
    age_H_start = path_ages_dict['H'][0]
    
    # Calendar time gap should match age2 gap roughly
    pm.Potential('path_alignment',
                pm.logp(pm.Normal.dist(mu=age2_gap, sigma=1.0),
                       age_H_start - age_I_end))
    
    # ------------------------------------------------------------------
    # Isotope likelihood - simplified
    # ------------------------------------------------------------------
    
    d13c_noise = pm.HalfNormal('d13c_noise', sigma=0.5)
    d13c_mean = pm.Normal('d13c_mean', mu=-2.0, sigma=0.5)
    
    pm.Normal('d13c_obs',
             mu=d13c_mean,
             sigma=d13c_noise,
             observed=df_test['d13c'].values)
    
    d18o_noise = pm.HalfNormal('d18o_noise', sigma=0.3)
    d18o_mean = pm.Normal('d18o_mean', mu=25.4, sigma=0.3)
    
    pm.Normal('d18o_obs',
             mu=d18o_mean,
             sigma=d18o_noise,
             observed=df_test['d18o'].values)


# ============================================================================
# STEP 3: Sample from the model
# ============================================================================

print("\nStarting MCMC sampling...")
print("This may take a few minutes...\n")

with simple_model:
    # Prior predictive check
    prior_pred = pm.sample_prior_predictive(samples=500, random_seed=42)
    print("✓ Prior predictive sampling successful")
    
    # Sample from posterior
    trace = pm.sample(
        draws=1000,
        tune=500,
        cores=2,
        target_accept=0.9,
        random_seed=42,
        return_inferencedata=True
    )
    print("\n✓ Posterior sampling successful")

# ============================================================================
# STEP 4: Analyze results
# ============================================================================

print("\n" + "="*80)
print("RESULTS")
print("="*80)

# Extract ages
ages_I_posterior = trace.posterior['ages_I'].values
ages_H_posterior = trace.posterior['ages_H'].values

# Flatten
ages_I_flat = ages_I_posterior.reshape(-1, ages_I_posterior.shape[-1])
ages_H_flat = ages_H_posterior.reshape(-1, ages_H_posterior.shape[-1])

# Get statistics
ages_I_median = np.median(ages_I_flat, axis=0)
ages_I_lower = np.percentile(ages_I_flat, 2.5, axis=0)
ages_I_upper = np.percentile(ages_I_flat, 97.5, axis=0)

ages_H_median = np.median(ages_H_flat, axis=0)
ages_H_lower = np.percentile(ages_H_flat, 2.5, axis=0)
ages_H_upper = np.percentile(ages_H_flat, 97.5, axis=0)

# Combine
ages_median = np.concatenate([ages_I_median, ages_H_median])
ages_lower = np.concatenate([ages_I_lower, ages_H_lower])
ages_upper = np.concatenate([ages_I_upper, ages_H_upper])

# Add to dataframe
df_test['age_median'] = ages_median
df_test['age_lower'] = ages_lower
df_test['age_upper'] = ages_upper
df_test['age_uncertainty'] = ages_upper - ages_lower

print(f"\nAge estimates for paths {test_paths}:")
print(f"  Oldest sample: {ages_median.min():.2f} (±{df_test.iloc[ages_median.argmin()]['age_uncertainty']/2:.2f})")
print(f"  Youngest sample: {ages_median.max():.2f} (±{df_test.iloc[ages_median.argmax()]['age_uncertainty']/2:.2f})")
print(f"  Total span: {ages_median.max() - ages_median.min():.2f} years")
print(f"  Mean uncertainty: ±{df_test['age_uncertainty'].mean()/2:.2f} years")

# Check that ages increase
ages_increasing = np.all(np.diff(ages_median) > 0)
print(f"\nAges monotonically increasing: {ages_increasing}")
if not ages_increasing:
    print("  Warning: Some ages decrease! Check model.")

# Dark band timing
band_month_posterior = trace.posterior['band_month'].values.flatten()
print(f"\nDark band timing:")
print(f"  Forms around month: {np.median(band_month_posterior):.1f} ± {np.std(band_month_posterior):.1f}")
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
month_idx = int(np.median(band_month_posterior))
if 0 <= month_idx < 12:
    print(f"  Approximately: {month_names[month_idx]}")

# Check convergence
print("\nConvergence diagnostics:")
rhat = az.rhat(trace)
for var in ['mean_time_per_sample', 'band_month']:
    if var in rhat.data_vars:
        print(f"  {var}: R-hat = {float(rhat[var].values):.4f}")

# ============================================================================
# STEP 5: Visualize
# ============================================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Age estimates
ax1 = axes[0, 0]
for path in test_paths:
    mask = df_test['path_name'] == path
    path_data = df_test[mask]
    
    ax1.fill_between(path_data.index,
                     path_data['age_lower'],
                     path_data['age_upper'],
                     alpha=0.3, label=f'Path {path} (95% CI)')
    ax1.plot(path_data.index, path_data['age_median'], 'o-', markersize=3)
    
    # Mark dark bands
    bands = path_data[path_data['age1'].notna()]
    ax1.scatter(bands.index, bands['age_median'], s=100, marker='|',
               color='red', linewidths=2, zorder=10)

ax1.set_xlabel('Sample Index (Older → Younger)')
ax1.set_ylabel('Estimated Age (years CE)')
ax1.set_title('Age Model with Uncertainty\n(Red bars = dark band starts)')
ax1.legend()
ax1.grid(alpha=0.3)

# Plot 2: d13C vs age
ax2 = axes[0, 1]
colors = {'I': 'blue', 'H': 'orange'}
for path in test_paths:
    mask = df_test['path_name'] == path
    ax2.scatter(df_test.loc[mask, 'age_median'], 
               df_test.loc[mask, 'd13c'],
               alpha=0.6, s=30, label=f'Path {path}', color=colors[path])

ax2.set_xlabel('Estimated Age (years CE)')
ax2.set_ylabel('δ¹³C (‰)')
ax2.set_title('δ¹³C vs Age')
ax2.legend()
ax2.grid(alpha=0.3)

# Plot 3: d18O vs age
ax3 = axes[1, 0]
for path in test_paths:
    mask = df_test['path_name'] == path
    ax3.scatter(df_test.loc[mask, 'age_median'],
               df_test.loc[mask, 'd18o'],
               alpha=0.6, s=30, label=f'Path {path}', color=colors[path])

ax3.set_xlabel('Estimated Age (years CE)')
ax3.set_ylabel('δ¹⁸O (‰)')
ax3.set_title('δ¹⁸O vs Age')
ax3.legend()
ax3.grid(alpha=0.3)

# Plot 4: Dark band seasonal timing
ax4 = axes[1, 1]
ax4.hist(band_month_posterior, bins=30, alpha=0.7, edgecolor='black')
ax4.axvline(np.median(band_month_posterior), color='red',
           linestyle='--', linewidth=2, label='Median')
ax4.set_xlabel('Month of Year')
ax4.set_ylabel('Posterior Density')
ax4.set_title('Dark Band Seasonal Timing')
ax4.set_xticks(np.arange(0, 13, 1))
ax4.set_xticklabels([''] + month_names, rotation=45)
ax4.legend()
ax4.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('corrected_age_model_results.png', dpi=150, bbox_inches='tight')
print("\n✓ Results figure saved")

# Save results
df_test.to_csv('age_estimates_corrected.csv', index=False)
print("✓ Age estimates saved to CSV")

# Summary
summary = az.summary(trace, var_names=['mean_time_per_sample', 'band_month', 
                                        'age_I_start', 'age_H_start'])
print("\nParameter estimates:")
print(summary)

print("\n" + "="*80)
print("CORRECTED MODEL COMPLETE")
print("="*80)
print("\nKey corrections made:")
print("1. Temporal direction: FORWARD in time (adding increments)")
print("2. Path order: I (older) before H (younger)")
print("3. Age differences: ages[i+1] - ages[i] should be positive")
print("4. Collection anchored at 2007, working backwards in age2")

In [ ]:
df = pd.read_excel('../../../2025-12-20_AgeModel-NewXrayPatch/age_model/lab_results.xlsx')


In [ ]:
"""
CORRECTED Bayesian Age Model for Coral Data
============================================

IMPORTANT TEMPORAL STRUCTURE:
- Paths in ALPHABETICAL order: O (oldest) → H (youngest)
- Within each path: path_num 1 (oldest) → higher numbers (younger)
- Data rows: index 0 (oldest, ~1928) → index 573 (youngest, 2007)
- age2: 0.05 (oldest) → 79.1 (youngest)

Collection date: April 2007
Oldest sample: ~1928 (79 years of record)
"""

import pymc as pm
import numpy as np
import pandas as pd
import arviz as az
import matplotlib.pyplot as plt
from scipy import stats
import pytensor.tensor as pt

# Load data
df = pd.read_excel('../../../2025-12-20_AgeModel-NewXrayPatch/age_model/lab_results.xlsx')

print("="*80)
print("CORRECTED BAYESIAN AGE MODEL")
print("="*80)
print(f"Total samples: {len(df)}")
print(f"Paths (oldest→youngest): {' → '.join(sorted(df['path_name'].unique()))}")
print(f"Timeline: ~1928 (age2=0.05) to 2007 (age2=79.1)")
print("="*80)

# ============================================================================
# STEP 1: Test with just TWO paths
# ============================================================================

# Use paths H and I (youngest, most recent - easier to validate)
test_paths = ['I', 'H']
df_test = df[df['path_name'].isin(test_paths)].copy()
df_test = df_test.reset_index(drop=True)

print(f"\nWorking with paths {test_paths} (youngest paths)")
print(f"Number of samples: {len(df_test)}")
print(f"Number of dark bands: {df_test['age1'].notna().sum()}")
print(f"age2 range: {df_test['age2'].min():.2f} to {df_test['age2'].max():.2f}")

# Collection date
collection_date = 2007.25  # April 2007

# ============================================================================
# STEP 2: Build Age Model with CORRECT temporal direction
# ============================================================================

with pm.Model() as simple_model:
    
    # ------------------------------------------------------------------
    # Prior: Mean time per sample
    # ------------------------------------------------------------------
    # Paths H and I together: ~235 samples spanning ~35 years
    # → ~7 samples per year → ~0.14 years per sample
    
    mean_time_per_sample = pm.Gamma('mean_time_per_sample', 
                                     alpha=14, beta=100)
    # Mean ≈ 0.14 year per sample
    
    # Reduced variation to encourage monotonicity
    variation = pm.HalfNormal('time_variation', sigma=0.03)
    # Tighter prior = more consistent increments = less chance of age reversals
    
    # ------------------------------------------------------------------
    # Build ages for each path (BACKWARD from known endpoint)
    # ------------------------------------------------------------------
    
    # CRITICAL: We KNOW the collection date at the youngest sample
    # This is Sample H-113 (last row of dataset)
    # So we should build ages BACKWARDS from this known point
    
    path_ages_dict = {}
    
    for path_name in sorted(test_paths, reverse=True):  # H before I (build from youngest)
        path_mask = df_test['path_name'] == path_name
        path_data = df_test[path_mask]
        n_samples = len(path_data)
        
        if path_name == 'H':  # Youngest path - ANCHOR HERE
            # Last sample (highest path_num) is at collection date
            # We build ages BACKWARDS from this known point
            last_age = pm.Normal('age_H_end',
                                mu=collection_date,
                                sigma=0.01)  # Very tight - we KNOW this!
            
            # Time increments (NEGATIVE because going BACKWARDS in the data)
            # path_num decreases = age decreases
            time_increments = pm.Lognormal(f'time_inc_{path_name}',
                                          mu=np.log(mean_time_per_sample),
                                          sigma=variation,
                                          shape=n_samples-1)
            
            # Build ages BACKWARDS from last sample
            # Reverse cumsum to go from end to beginning
            ages_reversed = last_age - pt.concatenate([[0], pt.cumsum(time_increments)])
            
            # Flip to match data order (oldest to youngest)
            ages = pm.Deterministic(f'ages_{path_name}',
                                   ages_reversed[::-1])
            
        else:  # Path I (older)
            # Path I ends where Path H begins (roughly)
            # Get the first (oldest) sample age from path H
            first_age = pm.Normal('age_I_start',
                                 mu=collection_date - 18.0,
                                 sigma=0.5)
            
            # Time increments (POSITIVE going forward through path I)
            time_increments = pm.Lognormal(f'time_inc_{path_name}',
                                          mu=np.log(mean_time_per_sample),
                                          sigma=variation,
                                          shape=n_samples-1)
            
            ages = pm.Deterministic(f'ages_{path_name}',
                                   first_age + pt.concatenate([[0], 
                                                               pt.cumsum(time_increments)]))
        
        path_ages_dict[path_name] = ages
    
    # Concatenate all ages (I first, then H)
    all_ages = pm.math.concatenate([path_ages_dict['I'], path_ages_dict['H']])
    
    # ------------------------------------------------------------------
    # Dark band seasonal constraint (CIRCULAR STATISTICS)
    # ------------------------------------------------------------------
    
    # TEMPORARILY DISABLED - will add back once basic model works
    # The VonMises constraint is causing -inf during initialization
    # Let's get the basic age model working first
    
    # When in the year do dark bands form? (in radians, 0-2π)
    band_month_rad = pm.Uniform('band_month_rad', lower=0, upper=2*np.pi)
    band_concentration = pm.Gamma('band_concentration', alpha=5, beta=1)
    
    # Get samples with dark band markers
    band_indices = df_test[df_test['age1'].notna()].index.values
    
    # COMMENTING OUT VONMISES FOR NOW - WILL ADD BACK LATER
    # if len(band_indices) > 0:
    #     band_ages = all_ages[band_indices]
    #     band_months_frac = band_ages % 1.0
    #     band_months_rad = band_months_frac * 2 * np.pi
    #     band_months_rad_wrapped = band_months_rad % (2 * np.pi)
    #     
    #     pm.Potential('band_timing',
    #                 pm.logp(pm.VonMises.dist(mu=band_month_rad,
    #                                         kappa=band_concentration_clipped),
    #                        band_months_rad_wrapped).sum())
    
    # ------------------------------------------------------------------
    # Dark bands should be roughly annual
    # ------------------------------------------------------------------
    
    if len(band_indices) > 1:
        for i in range(len(band_indices) - 1):
            # Bands should be ~1 year apart (ages increase through dataset)
            age_diff = all_ages[band_indices[i+1]] - all_ages[band_indices[i]]
            
            # Use Potential instead of observed
            pm.Potential(f'annual_spacing_{i}',
                        pm.logp(pm.Normal.dist(mu=1.0, sigma=0.15), age_diff))
    
    # ------------------------------------------------------------------
    # MONOTONICITY CONSTRAINT: Ages should increase
    # ------------------------------------------------------------------
    
    # Calculate all age differences (should all be positive)
    age_diffs = all_ages[1:] - all_ages[:-1]
    
    # Soft constraint using Exponential distribution (only positive values)
    # But with a large threshold to allow initialization
    # Exponential naturally encourages positive values
    pm.Potential('monotonic_ages',
                pm.logp(pm.Exponential.dist(lam=1.0/0.14), 
                       pm.math.maximum(age_diffs, 0.001)).sum())
    # maximum(age_diffs, 0.001) prevents -inf if any diff is negative
    # but still penalizes small/negative differences
    
    # ------------------------------------------------------------------
    # Path alignment constraint
    # ------------------------------------------------------------------
    
    # Path I ends around age2=61.7, Path H starts around age2=60.7
    # They overlap, so constraint ensures smooth transition
    
    # Get age2 values at transition
    i_last_age2 = df_test[df_test['path_name'] == 'I']['age2'].max()
    h_first_age2 = df_test[df_test['path_name'] == 'H']['age2'].min()
    age2_gap = h_first_age2 - i_last_age2  # About -1 (they overlap)
    
    # Last sample of path I should connect reasonably to first of path H
    age_I_end = path_ages_dict['I'][-1]
    age_H_start = path_ages_dict['H'][0]
    
    # Calendar time gap should match age2 gap roughly
    pm.Potential('path_alignment',
                pm.logp(pm.Normal.dist(mu=age2_gap, sigma=1.0),
                       age_H_start - age_I_end))
    
    # ------------------------------------------------------------------
    # Isotope likelihood - simplified
    # ------------------------------------------------------------------
    
    d13c_noise = pm.HalfNormal('d13c_noise', sigma=0.5)
    d13c_mean = pm.Normal('d13c_mean', mu=-2.0, sigma=0.5)
    
    pm.Normal('d13c_obs',
             mu=d13c_mean,
             sigma=d13c_noise,
             observed=df_test['d13c'].values)
    
    d18o_noise = pm.HalfNormal('d18o_noise', sigma=0.3)
    d18o_mean = pm.Normal('d18o_mean', mu=25.4, sigma=0.3)
    
    pm.Normal('d18o_obs',
             mu=d18o_mean,
             sigma=d18o_noise,
             observed=df_test['d18o'].values)


# ============================================================================
# STEP 3: Sample from the model (WITH MONOTONICITY CONSTRAINT)
# ============================================================================

print("\nStarting MCMC sampling...")
print("This may take a few minutes...\n")
print("NOTE: VonMises band timing constraint temporarily disabled")
print("      Added explicit monotonicity constraint for ages.\n")

with simple_model:
    # Sample from posterior
    # Increased settings for better convergence
    trace = pm.sample(
        draws=1000,      # Increased from 500
        tune=1000,       # Increased from 250  
        chains=2,        # Will get warning but okay for now
        cores=1,         # Single core to avoid OpenMP warnings
        target_accept=0.95,  # Higher to reduce divergences
        random_seed=42,
        return_inferencedata=True,
        init='adapt_diag'
    )
    print("\n✓ Posterior sampling successful")

# ============================================================================
# STEP 4: Analyze results
# ============================================================================

def month_from_decimal_year(t):
    """Convert decimal year to month (1-12)"""
    frac = t - np.floor(t)
    return (np.floor(frac * 12).astype(int) % 12) + 1

print("\n" + "="*80)
print("RESULTS")
print("="*80)

# Extract ages
ages_I_posterior = trace.posterior['ages_I'].values
ages_H_posterior = trace.posterior['ages_H'].values

# Flatten
ages_I_flat = ages_I_posterior.reshape(-1, ages_I_posterior.shape[-1])
ages_H_flat = ages_H_posterior.reshape(-1, ages_H_posterior.shape[-1])

# Get statistics
ages_I_median = np.median(ages_I_flat, axis=0)
ages_I_lower = np.percentile(ages_I_flat, 2.5, axis=0)
ages_I_upper = np.percentile(ages_I_flat, 97.5, axis=0)

ages_H_median = np.median(ages_H_flat, axis=0)
ages_H_lower = np.percentile(ages_H_flat, 2.5, axis=0)
ages_H_upper = np.percentile(ages_H_flat, 97.5, axis=0)

# Combine
ages_median = np.concatenate([ages_I_median, ages_H_median])
ages_lower = np.concatenate([ages_I_lower, ages_H_lower])
ages_upper = np.concatenate([ages_I_upper, ages_H_upper])

# Add to dataframe
df_test['age_median'] = ages_median
df_test['age_lower'] = ages_lower
df_test['age_upper'] = ages_upper
df_test['age_uncertainty'] = ages_upper - ages_lower

print(f"\nAge estimates for paths {test_paths}:")
print(f"  Oldest sample: {ages_median.min():.2f} (±{df_test.iloc[ages_median.argmin()]['age_uncertainty']/2:.2f})")
print(f"  Youngest sample: {ages_median.max():.2f} (±{df_test.iloc[ages_median.argmax()]['age_uncertainty']/2:.2f})")
print(f"  Total span: {ages_median.max() - ages_median.min():.2f} years")
print(f"  Mean uncertainty: ±{df_test['age_uncertainty'].mean()/2:.2f} years")

# Check that ages increase
ages_increasing = np.all(np.diff(ages_median) > 0)
print(f"\nAges monotonically increasing: {ages_increasing}")

if not ages_increasing:
    # Find where ages decrease
    age_diffs_check = np.diff(ages_median)
    decreasing_indices = np.where(age_diffs_check <= 0)[0]
    
    print(f"  ⚠️  Found {len(decreasing_indices)} non-monotonic transitions:")
    for idx in decreasing_indices[:5]:  # Show first 5
        print(f"     Sample {idx} → {idx+1}: {ages_median[idx]:.3f} → {ages_median[idx+1]:.3f} (diff: {age_diffs_check[idx]:.4f})")
    if len(decreasing_indices) > 5:
        print(f"     ... and {len(decreasing_indices)-5} more")
    
    print(f"\n  Magnitude of violations:")
    print(f"     Most negative: {age_diffs_check.min():.4f} years")
    print(f"     Mean of negative diffs: {age_diffs_check[age_diffs_check <= 0].mean():.4f} years")
else:
    age_diffs_check = np.diff(ages_median)
    print(f"  ✅ All {len(age_diffs_check)} age increments are positive")
    print(f"     Min increment: {age_diffs_check.min():.4f} years")
    print(f"     Max increment: {age_diffs_check.max():.4f} years")
    print(f"     Mean increment: {age_diffs_check.mean():.4f} years")

# Dark band timing (will be NaN since constraint is disabled)
print(f"\nDark band timing: [Temporarily disabled - will add back later]")
print(f"  Number of dark bands identified: {len(band_indices)}")

# We'll analyze the actual observed band timing from the age estimates
if len(band_indices) > 0:
    band_ages_obs = ages_median[band_indices]
    band_months_obs = month_from_decimal_year(band_ages_obs)
    
    # Calculate circular mean and std
    angles = (band_months_obs - 1) * 2 * np.pi / 12
    sin_mean = np.mean(np.sin(angles))
    cos_mean = np.mean(np.cos(angles))
    mean_angle = np.arctan2(sin_mean, cos_mean)
    mean_month_circ = (mean_angle * 12 / (2 * np.pi)) % 12 + 1
    
    # Circular std
    R = np.sqrt(sin_mean**2 + cos_mean**2)
    std_rad = np.sqrt(-2 * np.log(R)) if R > 1e-10 else np.pi
    std_month_circ = std_rad * 12 / (2 * np.pi)
    
    month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                   'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
    month_idx = int(np.round(mean_month_circ)) - 1
    
    print(f"  Observed band timing from ages: ~{month_names[month_idx]}")
    print(f"  Circular mean month: {mean_month_circ:.1f} ± {std_month_circ:.1f}")
else:
    mean_month_circ = 6  # Default for plotting
    std_month_circ = 2.0  # Default
    month_idx = 5
    month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                   'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

# Check convergence
print("\nConvergence diagnostics:")
rhat = az.rhat(trace)
for var in ['mean_time_per_sample', 'band_month']:
    if var in rhat.data_vars:
        print(f"  {var}: R-hat = {float(rhat[var].values):.4f}")

# ============================================================================
# STEP 5: Visualize (with circular statistics)
# ============================================================================

def month_from_decimal_year(t):
    """Convert decimal year to month (1-12)"""
    frac = t - np.floor(t)
    return (np.floor(frac * 12).astype(int) % 12) + 1

fig = plt.figure(figsize=(16, 10))
gs = fig.add_gridspec(2, 3, hspace=0.3, wspace=0.3)

# Plot 1: Age estimates
ax1 = fig.add_subplot(gs[0, :2])
for path in test_paths:
    mask = df_test['path_name'] == path
    path_data = df_test[mask]
    
    ax1.fill_between(path_data.index,
                     path_data['age_lower'],
                     path_data['age_upper'],
                     alpha=0.3, label=f'Path {path} (95% CI)')
    ax1.plot(path_data.index, path_data['age_median'], 'o-', markersize=3)
    
    # Mark dark bands
    bands = path_data[path_data['age1'].notna()]
    ax1.scatter(bands.index, bands['age_median'], s=100, marker='|',
               color='red', linewidths=2, zorder=10)

ax1.set_xlabel('Sample Index (Older → Younger)')
ax1.set_ylabel('Estimated Age (years CE)')
ax1.set_title('Age Model with Uncertainty\n(Red bars = dark band starts)')
ax1.legend()
ax1.grid(alpha=0.3)

# Plot 2: ROSE DIAGRAM for dark band timing
ax2 = fig.add_subplot(gs[0, 2], projection='polar')

# Get observed band months
if len(band_indices) > 0:
    band_ages_obs = df_test.loc[band_indices, 'age_median'].values
    observed_band_months = month_from_decimal_year(band_ages_obs)

    # Create histogram for rose diagram (all 12 months)
    counts, bin_edges = np.histogram(observed_band_months, 
                                     bins=np.arange(0.5, 13.5, 1))

    # Angles for each month (FIXED: cover all 12 months)
    # Month 1 (Jan) at top (90°), going clockwise
    # Need to map: Jan(1)→90°, Feb(2)→60°, Mar(3)→30°, Apr(4)→0°, etc.
    theta = np.linspace(0, 2*np.pi, 12, endpoint=False)
    # Rotate so January is at top (subtract π/2)
    theta = theta + np.pi/2
    
    width = 2 * np.pi / 12  # Width of each bar

    # Plot bars for ALL 12 months (even if count is 0)
    bars = ax2.bar(theta, counts, width=width, color='steelblue',
                   alpha=0.7, edgecolor='black', linewidth=1.5)

    # Add mean direction arrow
    mean_theta = mean_angle + np.pi/2  # Rotate to match our theta convention
    max_count = np.max(counts) if np.max(counts) > 0 else 1
    ax2.arrow(mean_theta, 0, 0, max_count * 0.8,
             head_width=0.3, head_length=max_count * 0.15,
             fc='red', ec='red', linewidth=3, alpha=0.8, zorder=10)

    # Labels - all 12 months
    month_names_short = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                         'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
    ax2.set_xticks(theta)
    ax2.set_xticklabels(month_names_short, fontsize=9)
    ax2.set_theta_direction(-1)  # Clockwise
    ax2.set_theta_zero_location('N')  # North at top
    ax2.set_title(f'Dark Band Seasonal Timing\n{month_names[month_idx]} ± {std_month_circ:.1f} mo',
                 pad=20, fontweight='bold')
    ax2.grid(True, alpha=0.3)
else:
    ax2.text(0.5, 0.5, 'No dark bands found',
            ha='center', va='center', transform=ax2.transAxes)

# Plot 3: d13C vs age
ax3 = fig.add_subplot(gs[1, 0])
colors = {'I': 'blue', 'H': 'orange'}
for path in test_paths:
    mask = df_test['path_name'] == path
    ax3.scatter(df_test.loc[mask, 'age_median'], 
               df_test.loc[mask, 'd13c'],
               alpha=0.6, s=30, label=f'Path {path}', color=colors[path])

ax3.set_xlabel('Estimated Age (years CE)')
ax3.set_ylabel('δ¹³C (‰)')
ax3.set_title('δ¹³C vs Age')
ax3.legend()
ax3.grid(alpha=0.3)

# Plot 4: d18O vs age
ax4 = fig.add_subplot(gs[1, 1])
for path in test_paths:
    mask = df_test['path_name'] == path
    ax4.scatter(df_test.loc[mask, 'age_median'],
               df_test.loc[mask, 'd18o'],
               alpha=0.6, s=30, label=f'Path {path}', color=colors[path])

ax4.set_xlabel('Estimated Age (years CE)')
ax4.set_ylabel('δ¹⁸O (‰)')
ax4.set_title('δ¹⁸O vs Age')
ax4.legend()
ax4.grid(alpha=0.3)

# Plot 5: Band month distribution (from observed ages, not posterior)
ax5 = fig.add_subplot(gs[1, 2])

if len(band_indices) > 0:
    # Get observed band months from estimated ages
    band_ages_plot = df_test.loc[band_indices, 'age_median'].values
    observed_band_months = month_from_decimal_year(band_ages_plot)
    
    ax5.hist(observed_band_months, bins=np.arange(0.5, 13.5, 1),
            alpha=0.7, edgecolor='black', color='steelblue')
    ax5.axvline(mean_month_circ, color='red', linestyle='--',
               linewidth=2, label=f'Circ. Mean: {mean_month_circ:.1f}')
    ax5.set_xlabel('Month')
    ax5.set_ylabel('Frequency')
    ax5.set_title('Dark Band Timing\n(from estimated ages)')
    ax5.set_xticks(range(1, 13))
    ax5.set_xticklabels(month_names_short, rotation=45)
    ax5.legend()
    ax5.grid(alpha=0.3, axis='y')
else:
    ax5.text(0.5, 0.5, 'No dark bands', ha='center', va='center',
            transform=ax5.transAxes)

plt.tight_layout()
plt.savefig('corrected_age_model_results.png', dpi=150, bbox_inches='tight')
print("\n✓ Results figure saved")

# Save results
df_test.to_csv('age_estimates_corrected_without_climate.csv', index=False)
print("✓ Age estimates saved to CSV")

# Summary statistics (only variables that exist in the model)
summary = az.summary(trace, var_names=['mean_time_per_sample', 'age_I_start', 'age_H_end'])
print("\nParameter estimates:")
print(summary)

print("\n" + "="*80)
print("SIMPLIFIED MODEL RESULTS")
print("="*80)
print("\nKey findings:")
print(f"1. Age span: {ages_median.min():.1f} to {ages_median.max():.1f} CE (~{ages_median.max() - ages_median.min():.1f} years)")
print(f"2. Dark bands occur primarily: {month_names[month_idx]} (circular mean: {mean_month_circ:.1f} ± {std_month_circ:.1f} months)")
print(f"3. Mean age uncertainty: ±{df_test['age_uncertainty'].mean()/2:.2f} years")
print(f"4. Youngest sample uncertainty: ±{df_test['age_uncertainty'].iloc[-1]/2:.2f} years (anchored at 2007)")
print(f"5. Oldest sample uncertainty: ±{df_test['age_uncertainty'].iloc[0]/2:.2f} years")

if not ages_increasing:
    print("\n⚠️  WARNING: Ages not strictly monotonic!")
    print("   Some samples have ages that decrease.")
    print("   Possible causes:")
    print("   - Time increment priors too variable")
    print("   - Annual spacing constraints too weak")  
    print("   - Need stronger monotonicity constraint")
    print("   Action: Consider adding explicit monotonicity constraint")

print("\n✅ Next steps:")
print("1. Review the age estimates and rose diagram")
print("2. If reasonable, extend to all 8 paths")
print("3. Add VonMises constraint if tighter seasonal clustering needed")
print("4. Add SST/SSS calibration for 1960-2007 period")
print("5. Implement δ¹³C trend decomposition (seasonal vs long-term)")

In [ ]:
"""
CORRECTED Bayesian Age Model for Coral Data
============================================

IMPORTANT TEMPORAL STRUCTURE:
- Paths in ALPHABETICAL order: O (oldest) → H (youngest)
- Within each path: path_num 1 (oldest) → higher numbers (younger)
- Data rows: index 0 (oldest, ~1928) → index 573 (youngest, 2007)
- age2: 0.05 (oldest) → 79.1 (youngest)

Collection date: April 2007
Oldest sample: ~1928 (79 years of record)
"""

import pymc as pm
import numpy as np
import pandas as pd
import arviz as az
import matplotlib.pyplot as plt
from scipy import stats
import pytensor.tensor as pt

# Load data
df = pd.read_excel('../../../2025-12-20_AgeModel-NewXrayPatch/age_model/lab_results.xlsx')

# Load climate data
df_climate = pd.read_csv('../../../2026-01-15_Climate-Model/analysis/climate_df.csv')
df_climate = df_climate.sort_values('age_absolute').reset_index(drop=True)

print("="*80)
print("CORRECTED BAYESIAN AGE MODEL WITH CLIMATE CALIBRATION")
print("="*80)
print(f"Total coral samples: {len(df)}")
print(f"Climate data: {df_climate['age_absolute'].min():.1f} to {df_climate['age_absolute'].max():.1f} ({len(df_climate)} months)")
print(f"\nClimate data insight:")
print(f"  synthetic_d18o = -0.22*SST + 0.80*SSS - 21.12  (R²=1.0)")
print(f"  SST range: {df_climate['SST'].min():.1f} to {df_climate['SST'].max():.1f} °C")
print(f"  SSS range: {df_climate['SSS'].min():.2f} to {df_climate['SSS'].max():.2f} psu")
print(f"  Calibration period (coral-climate overlap): ~1958-2007 (~49 years)")

# ============================================================================
# STEP 1: Test with just TWO paths
# ============================================================================

# Use paths H and I (youngest, most recent - easier to validate)
test_paths = ['I', 'H']
df_test = df[df['path_name'].isin(test_paths)].copy()
df_test = df_test.reset_index(drop=True)

print(f"\nWorking with paths {test_paths} (youngest paths)")
print(f"Number of samples: {len(df_test)}")
print(f"Number of dark bands: {df_test['age1'].notna().sum()}")
print(f"age2 range: {df_test['age2'].min():.2f} to {df_test['age2'].max():.2f}")

# Collection date
collection_date = 2007.25  # April 2007

# ============================================================================
# STEP 2: Build Age Model with CORRECT temporal direction
# ============================================================================

with pm.Model() as simple_model:
    
    # ------------------------------------------------------------------
    # Prior: Mean time per sample
    # ------------------------------------------------------------------
    # Paths H and I together: ~235 samples spanning ~35 years
    # → ~7 samples per year → ~0.14 years per sample
    
    mean_time_per_sample = pm.Gamma('mean_time_per_sample', 
                                     alpha=14, beta=100)
    # Mean ≈ 0.14 year per sample
    
    # Reduced variation to encourage monotonicity
    variation = pm.HalfNormal('time_variation', sigma=0.03)
    # Tighter prior = more consistent increments = less chance of age reversals
    
    # ------------------------------------------------------------------
    # Build ages for each path (BACKWARD from known endpoint)
    # ------------------------------------------------------------------
    
    # CRITICAL: We KNOW the collection date at the youngest sample
    # This is Sample H-113 (last row of dataset)
    # So we should build ages BACKWARDS from this known point
    
    path_ages_dict = {}
    
    for path_name in sorted(test_paths, reverse=True):  # H before I (build from youngest)
        path_mask = df_test['path_name'] == path_name
        path_data = df_test[path_mask]
        n_samples = len(path_data)
        
        if path_name == 'H':  # Youngest path - ANCHOR HERE
            # Last sample (highest path_num) is at collection date
            # We build ages BACKWARDS from this known point
            last_age = pm.Normal('age_H_end',
                                mu=collection_date,
                                sigma=0.05)  # Very tight - we KNOW this!
            
            # Time increments (NEGATIVE because going BACKWARDS in the data)
            # path_num decreases = age decreases
            time_increments = pm.Lognormal(f'time_inc_{path_name}',
                                          mu=np.log(mean_time_per_sample),
                                          sigma=variation,
                                          shape=n_samples-1)
            
            # Build ages BACKWARDS from last sample
            # Reverse cumsum to go from end to beginning
            ages_reversed = last_age - pt.concatenate([[0], pt.cumsum(time_increments)])
            
            # Flip to match data order (oldest to youngest)
            ages = pm.Deterministic(f'ages_{path_name}',
                                   ages_reversed[::-1])
            
        else:  # Path I (older)
            # Path I ends where Path H begins (roughly)
            # Get the first (oldest) sample age from path H
            first_age = pm.Normal('age_I_start',
                                 mu=collection_date - 18.0,
                                 sigma=0.5)
            
            # Time increments (POSITIVE going forward through path I)
            time_increments = pm.Lognormal(f'time_inc_{path_name}',
                                          mu=np.log(mean_time_per_sample),
                                          sigma=variation,
                                          shape=n_samples-1)
            
            ages = pm.Deterministic(f'ages_{path_name}',
                                   first_age + pt.concatenate([[0], 
                                                               pt.cumsum(time_increments)]))
        
        path_ages_dict[path_name] = ages
    
    # Concatenate all ages (I first, then H)
    all_ages = pm.math.concatenate([path_ages_dict['I'], path_ages_dict['H']])
    
    # ------------------------------------------------------------------
    # Dark band seasonal constraint (CIRCULAR STATISTICS)
    # ------------------------------------------------------------------
    
    # TEMPORARILY DISABLED - will add back once basic model works
    # The VonMises constraint is causing -inf during initialization
    # Let's get the basic age model working first
    
    # When in the year do dark bands form? (in radians, 0-2π)
    band_month_rad = pm.Uniform('band_month_rad', lower=0, upper=2*np.pi)
    band_concentration = pm.Gamma('band_concentration', alpha=5, beta=1)
    
    # Get samples with dark band markers
    band_indices = df_test[df_test['age1'].notna()].index.values
    
    # COMMENTING OUT VONMISES FOR NOW - WILL ADD BACK LATER
    # if len(band_indices) > 0:
    #     band_ages = all_ages[band_indices]
    #     band_months_frac = band_ages % 1.0
    #     band_months_rad = band_months_frac * 2 * np.pi
    #     band_months_rad_wrapped = band_months_rad % (2 * np.pi)
    #     
    #     pm.Potential('band_timing',
    #                 pm.logp(pm.VonMises.dist(mu=band_month_rad,
    #                                         kappa=band_concentration_clipped),
    #                        band_months_rad_wrapped).sum())
    
    # ------------------------------------------------------------------
    # Dark bands should be roughly annual
    # ------------------------------------------------------------------
    
    if len(band_indices) > 1:
        for i in range(len(band_indices) - 1):
            # Bands should be ~1 year apart (ages increase through dataset)
            age_diff = all_ages[band_indices[i+1]] - all_ages[band_indices[i]]
            
            # Use Potential instead of observed
            pm.Potential(f'annual_spacing_{i}',
                        pm.logp(pm.Normal.dist(mu=1.0, sigma=0.15), age_diff))
    
    # ------------------------------------------------------------------
    # MONOTONICITY CONSTRAINT: Ages should increase
    # ------------------------------------------------------------------
    
    # Calculate all age differences (should all be positive)
    age_diffs = all_ages[1:] - all_ages[:-1]
    
    # Soft constraint using Exponential distribution (only positive values)
    # But with a large threshold to allow initialization
    # Exponential naturally encourages positive values
    pm.Potential('monotonic_ages',
                pm.logp(pm.Exponential.dist(lam=1.0/0.14), 
                       pm.math.maximum(age_diffs, 0.001)).sum())
    # maximum(age_diffs, 0.001) prevents -inf if any diff is negative
    # but still penalizes small/negative differences
    
    # ------------------------------------------------------------------
    # Path alignment constraint
    # ------------------------------------------------------------------
    
    # Path I ends around age2=61.7, Path H starts around age2=60.7
    # They overlap, so constraint ensures smooth transition
    
    # Get age2 values at transition
    i_last_age2 = df_test[df_test['path_name'] == 'I']['age2'].max()
    h_first_age2 = df_test[df_test['path_name'] == 'H']['age2'].min()
    age2_gap = h_first_age2 - i_last_age2  # About -1 (they overlap)
    
    # Last sample of path I should connect reasonably to first of path H
    age_I_end = path_ages_dict['I'][-1]
    age_H_start = path_ages_dict['H'][0]
    
    # Calendar time gap should match age2 gap roughly
    pm.Potential('path_alignment',
                pm.logp(pm.Normal.dist(mu=age2_gap, sigma=1.0),
                       age_H_start - age_I_end))
    
    # ------------------------------------------------------------------
    # δ¹⁸O LIKELIHOOD WITH CLIMATE CALIBRATION
    # ------------------------------------------------------------------
    #
    # KEY INSIGHT FROM DATA EXPLORATION:
    # synthetic_d18o = -0.22 * SST + 0.80 * SSS - 21.1165  (R²=1.0)
    # This is already pre-computed and captures the SST/SSS signal.
    #
    # Coral d18O is on absolute VPDB scale (~25-26‰)
    # Synthetic d18O is the SST/SSS anomaly scale (~-1 to +1‰)
    #
    # So the full model is:
    #   coral_d18O = d18o_offset + synthetic_d18o(age) + noise
    #
    # Where d18o_offset is the mean absolute d18O (~25.6‰)
    # And synthetic_d18o(age) is interpolated from climate data
    #
    # This links coral isotopes directly to temperature and salinity.
    # ------------------------------------------------------------------

    # Offset between absolute coral d18O and the synthetic anomaly
    d18o_offset = pm.Normal('d18o_offset', mu=25.6, sigma=0.3)
    # Prior: centered on observed coral d18O mean (~25.6‰)

    # Observation noise (analytical + vital effects)
    d18o_noise = pm.HalfNormal('d18o_noise', sigma=0.2)

    # Build d18O predictions for each sample
    # Strategy:
    #   - Calibration period (1958-2007): use climate data directly
    #   - Pre-climate period (<1958): use offset + noise only
    #
    # For the calibration period: interpolate synthetic_d18o
    # at each sample's estimated age → links ages to climate

    # Split samples into calibration vs pre-climate
    climate_start = df_climate['age_absolute'].min()  # 1958.04

    n_samples = len(df_test)
    d18o_obs_vals = df_test['d18o'].values

    # For interpolation: precompute climate age and synthetic_d18o arrays
    # (These are fixed numpy arrays, not PyMC variables)
    climate_ages_np = df_climate['age_absolute'].values
    synthetic_d18o_np = df_climate['synthetic_d18o'].values

    # Build expected d18O for each sample using its estimated age
    # We use pt.switch to handle calibration vs pre-climate period
    #
    # For calibration samples: use nearest climate month
    # This is done using a precomputed lookup via linear interpolation

    # Vectorized interpolation using pytensor
    # For each sample age, find synthetic_d18o by linear interpolation
    # on the fixed climate grid

    # Convert climate arrays to pytensor constants ONCE
    # so the whole interpolation stays in the computational graph
    climate_ages_pt   = pt.as_tensor_variable(climate_ages_np.astype('float64'))
    synthetic_d18o_pt = pt.as_tensor_variable(synthetic_d18o_np.astype('float64'))

    def interp_synthetic_d18o(sample_age):
        """
        Linearly interpolate synthetic_d18o at a given age.
        Stays entirely in PyTensor so gradients flow correctly.
        """
        # Find bracket index
        idx = pt.searchsorted(climate_ages_pt, sample_age) - 1
        idx = pt.clip(idx, 0, len(climate_ages_np) - 2)

        # Use PyTensor indexing (not numpy indexing)
        age_lo   = climate_ages_pt[idx]
        age_hi   = climate_ages_pt[idx + 1]
        synth_lo = synthetic_d18o_pt[idx]
        synth_hi = synthetic_d18o_pt[idx + 1]

        # Linear interpolation weight
        weight = (sample_age - age_lo) / (age_hi - age_lo + 1e-10)
        weight = pt.clip(weight, 0.0, 1.0)

        return synth_lo + weight * (synth_hi - synth_lo)

    # Compute expected d18O for ALL samples
    d18o_expected_list = []
    for i in range(n_samples):
        age_i = all_ages[i]

        # Is this sample in the calibration period?
        in_calibration = pt.ge(age_i, climate_start)

        # Expected d18O:
        # - In calibration: offset + interpolated synthetic_d18o
        # - Before climate data: just offset (no climate constraint)
        synth_val = interp_synthetic_d18o(age_i)
        expected_calib = d18o_offset + synth_val
        expected_pre   = d18o_offset  # no climate signal to match

        d18o_expected_list.append(
            pt.switch(in_calibration, expected_calib, expected_pre)
        )

    d18o_expected = pt.stack(d18o_expected_list)

    # Likelihood: observed coral d18O vs climate-predicted d18O
    pm.Potential('d18o_obs',
                pm.logp(pm.Normal.dist(mu=d18o_expected,
                                      sigma=d18o_noise),
                       d18o_obs_vals).sum())

    # ------------------------------------------------------------------
    # δ¹³C LIKELIHOOD (simplified - long-term trend preserved)
    # ------------------------------------------------------------------
    #
    # δ¹³C has a strong long-term trend in your data (your research focus).
    # We keep this simple for now: just a mean + noise.
    # Later: decompose into seasonal + long-term trend (GP or spline).
    # ------------------------------------------------------------------

    d13c_mean = pm.Normal('d13c_mean', mu=-2.0, sigma=0.5)
    d13c_noise = pm.HalfNormal('d13c_noise', sigma=0.5)

    pm.Normal('d13c_obs',
             mu=d13c_mean,
             sigma=d13c_noise,
             observed=df_test['d13c'].values)


# ============================================================================
# STEP 3: Sample from the model (WITH MONOTONICITY CONSTRAINT)
# ============================================================================

print("\nStarting MCMC sampling...")
print("This may take a few minutes...\n")
print("NOTE: VonMises band timing constraint temporarily disabled")
print("      Added explicit monotonicity constraint for ages.\n")

with simple_model:
    # Sample from posterior
    # Increased settings for better convergence
    trace = pm.sample(
        draws=1000,      # Increased from 500
        tune=1000,       # Increased from 250  
        chains=2,        # Will get warning but okay for now
        cores=1,         # Single core to avoid OpenMP warnings
        target_accept=0.95,  # Higher to reduce divergences
        random_seed=42,
        return_inferencedata=True,
        init='adapt_diag'
    )
    print("\n✓ Posterior sampling successful")

# ============================================================================
# STEP 4: Analyze results
# ============================================================================

def month_from_decimal_year(t):
    """Convert decimal year to month (1-12)"""
    frac = t - np.floor(t)
    return (np.floor(frac * 12).astype(int) % 12) + 1

print("\n" + "="*80)
print("RESULTS")
print("="*80)

# Extract ages
ages_I_posterior = trace.posterior['ages_I'].values
ages_H_posterior = trace.posterior['ages_H'].values

# Flatten
ages_I_flat = ages_I_posterior.reshape(-1, ages_I_posterior.shape[-1])
ages_H_flat = ages_H_posterior.reshape(-1, ages_H_posterior.shape[-1])

# Get statistics
ages_I_median = np.median(ages_I_flat, axis=0)
ages_I_lower = np.percentile(ages_I_flat, 2.5, axis=0)
ages_I_upper = np.percentile(ages_I_flat, 97.5, axis=0)

ages_H_median = np.median(ages_H_flat, axis=0)
ages_H_lower = np.percentile(ages_H_flat, 2.5, axis=0)
ages_H_upper = np.percentile(ages_H_flat, 97.5, axis=0)

# Combine
ages_median = np.concatenate([ages_I_median, ages_H_median])
ages_lower = np.concatenate([ages_I_lower, ages_H_lower])
ages_upper = np.concatenate([ages_I_upper, ages_H_upper])

# Add to dataframe
df_test['age_median'] = ages_median
df_test['age_lower'] = ages_lower
df_test['age_upper'] = ages_upper
df_test['age_uncertainty'] = ages_upper - ages_lower

print(f"\nAge estimates for paths {test_paths}:")
print(f"  Oldest sample: {ages_median.min():.2f} (±{df_test.iloc[ages_median.argmin()]['age_uncertainty']/2:.2f})")
print(f"  Youngest sample: {ages_median.max():.2f} (±{df_test.iloc[ages_median.argmax()]['age_uncertainty']/2:.2f})")
print(f"  Total span: {ages_median.max() - ages_median.min():.2f} years")
print(f"  Mean uncertainty: ±{df_test['age_uncertainty'].mean()/2:.2f} years")

# Check that ages increase
ages_increasing = np.all(np.diff(ages_median) > 0)
print(f"\nAges monotonically increasing: {ages_increasing}")

if not ages_increasing:
    # Find where ages decrease
    age_diffs_check = np.diff(ages_median)
    decreasing_indices = np.where(age_diffs_check <= 0)[0]
    
    print(f"  ⚠️  Found {len(decreasing_indices)} non-monotonic transitions:")
    for idx in decreasing_indices[:5]:  # Show first 5
        print(f"     Sample {idx} → {idx+1}: {ages_median[idx]:.3f} → {ages_median[idx+1]:.3f} (diff: {age_diffs_check[idx]:.4f})")
    if len(decreasing_indices) > 5:
        print(f"     ... and {len(decreasing_indices)-5} more")
    
    print(f"\n  Magnitude of violations:")
    print(f"     Most negative: {age_diffs_check.min():.4f} years")
    print(f"     Mean of negative diffs: {age_diffs_check[age_diffs_check <= 0].mean():.4f} years")
else:
    age_diffs_check = np.diff(ages_median)
    print(f"  ✅ All {len(age_diffs_check)} age increments are positive")
    print(f"     Min increment: {age_diffs_check.min():.4f} years")
    print(f"     Max increment: {age_diffs_check.max():.4f} years")
    print(f"     Mean increment: {age_diffs_check.mean():.4f} years")

# Dark band timing (will be NaN since constraint is disabled)
print(f"\nDark band timing: [Temporarily disabled - will add back later]")
print(f"  Number of dark bands identified: {len(band_indices)}")

# We'll analyze the actual observed band timing from the age estimates
if len(band_indices) > 0:
    band_ages_obs = ages_median[band_indices]
    band_months_obs = month_from_decimal_year(band_ages_obs)
    
    # Calculate circular mean and std
    angles = (band_months_obs - 1) * 2 * np.pi / 12
    sin_mean = np.mean(np.sin(angles))
    cos_mean = np.mean(np.cos(angles))
    mean_angle = np.arctan2(sin_mean, cos_mean)
    mean_month_circ = (mean_angle * 12 / (2 * np.pi)) % 12 + 1
    
    # Circular std
    R = np.sqrt(sin_mean**2 + cos_mean**2)
    std_rad = np.sqrt(-2 * np.log(R)) if R > 1e-10 else np.pi
    std_month_circ = std_rad * 12 / (2 * np.pi)
    
    month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                   'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
    month_idx = int(np.round(mean_month_circ)) - 1
    
    print(f"  Observed band timing from ages: ~{month_names[month_idx]}")
    print(f"  Circular mean month: {mean_month_circ:.1f} ± {std_month_circ:.1f}")
else:
    mean_month_circ = 6  # Default for plotting
    std_month_circ = 2.0  # Default
    month_idx = 5
    month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                   'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

# Check convergence
print("\nConvergence diagnostics:")
rhat = az.rhat(trace)
for var in ['mean_time_per_sample', 'band_month']:
    if var in rhat.data_vars:
        print(f"  {var}: R-hat = {float(rhat[var].values):.4f}")

# ============================================================================
# STEP 5: Visualize (with circular statistics)
# ============================================================================

def month_from_decimal_year(t):
    """Convert decimal year to month (1-12)"""
    frac = t - np.floor(t)
    return (np.floor(frac * 12).astype(int) % 12) + 1

fig = plt.figure(figsize=(16, 10))
gs = fig.add_gridspec(2, 3, hspace=0.3, wspace=0.3)

# Plot 1: Age estimates
ax1 = fig.add_subplot(gs[0, :2])
for path in test_paths:
    mask = df_test['path_name'] == path
    path_data = df_test[mask]
    
    ax1.fill_between(path_data.index,
                     path_data['age_lower'],
                     path_data['age_upper'],
                     alpha=0.3, label=f'Path {path} (95% CI)')
    ax1.plot(path_data.index, path_data['age_median'], 'o-', markersize=3)
    
    # Mark dark bands
    bands = path_data[path_data['age1'].notna()]
    ax1.scatter(bands.index, bands['age_median'], s=100, marker='|',
               color='red', linewidths=2, zorder=10)

ax1.set_xlabel('Sample Index (Older → Younger)')
ax1.set_ylabel('Estimated Age (years CE)')
ax1.set_title('Age Model with Uncertainty\n(Red bars = dark band starts)')
ax1.legend()
ax1.grid(alpha=0.3)

# Plot 2: ROSE DIAGRAM for dark band timing
ax2 = fig.add_subplot(gs[0, 2], projection='polar')

# Get observed band months
if len(band_indices) > 0:
    band_ages_obs = df_test.loc[band_indices, 'age_median'].values
    observed_band_months = month_from_decimal_year(band_ages_obs)

    # Create histogram for rose diagram (all 12 months)
    counts, bin_edges = np.histogram(observed_band_months, 
                                     bins=np.arange(0.5, 13.5, 1))

    # Angles for each month (FIXED: cover all 12 months)
    # Month 1 (Jan) at top (90°), going clockwise
    # Need to map: Jan(1)→90°, Feb(2)→60°, Mar(3)→30°, Apr(4)→0°, etc.
    theta = np.linspace(0, 2*np.pi, 12, endpoint=False)
    # Rotate so January is at top (subtract π/2)
    theta = theta + np.pi/2
    
    width = 2 * np.pi / 12  # Width of each bar

    # Plot bars for ALL 12 months (even if count is 0)
    bars = ax2.bar(theta, counts, width=width, color='steelblue',
                   alpha=0.7, edgecolor='black', linewidth=1.5)

    # Add mean direction arrow
    mean_theta = mean_angle + np.pi/2  # Rotate to match our theta convention
    max_count = np.max(counts) if np.max(counts) > 0 else 1
    ax2.arrow(mean_theta, 0, 0, max_count * 0.8,
             head_width=0.3, head_length=max_count * 0.15,
             fc='red', ec='red', linewidth=3, alpha=0.8, zorder=10)

    # Labels - all 12 months
    month_names_short = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                         'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
    ax2.set_xticks(theta)
    ax2.set_xticklabels(month_names_short, fontsize=9)
    ax2.set_theta_direction(-1)  # Clockwise
    ax2.set_theta_zero_location('N')  # North at top
    ax2.set_title(f'Dark Band Seasonal Timing\n{month_names[month_idx]} ± {std_month_circ:.1f} mo',
                 pad=20, fontweight='bold')
    ax2.grid(True, alpha=0.3)
else:
    ax2.text(0.5, 0.5, 'No dark bands found',
            ha='center', va='center', transform=ax2.transAxes)

# Plot 3: d13C vs age
ax3 = fig.add_subplot(gs[1, 0])
colors = {'I': 'blue', 'H': 'orange'}
for path in test_paths:
    mask = df_test['path_name'] == path
    ax3.scatter(df_test.loc[mask, 'age_median'], 
               df_test.loc[mask, 'd13c'],
               alpha=0.6, s=30, label=f'Path {path}', color=colors[path])

ax3.set_xlabel('Estimated Age (years CE)')
ax3.set_ylabel('δ¹³C (‰)')
ax3.set_title('δ¹³C vs Age')
ax3.legend()
ax3.grid(alpha=0.3)

# Plot 4: d18O vs age
ax4 = fig.add_subplot(gs[1, 1])
for path in test_paths:
    mask = df_test['path_name'] == path
    ax4.scatter(df_test.loc[mask, 'age_median'],
               df_test.loc[mask, 'd18o'],
               alpha=0.6, s=30, label=f'Path {path}', color=colors[path])

ax4.set_xlabel('Estimated Age (years CE)')
ax4.set_ylabel('δ¹⁸O (‰)')
ax4.set_title('δ¹⁸O vs Age')
ax4.legend()
ax4.grid(alpha=0.3)

# Plot 5: Band month distribution (from observed ages, not posterior)
ax5 = fig.add_subplot(gs[1, 2])

if len(band_indices) > 0:
    # Get observed band months from estimated ages
    band_ages_plot = df_test.loc[band_indices, 'age_median'].values
    observed_band_months = month_from_decimal_year(band_ages_plot)
    
    ax5.hist(observed_band_months, bins=np.arange(0.5, 13.5, 1),
            alpha=0.7, edgecolor='black', color='steelblue')
    ax5.axvline(mean_month_circ, color='red', linestyle='--',
               linewidth=2, label=f'Circ. Mean: {mean_month_circ:.1f}')
    ax5.set_xlabel('Month')
    ax5.set_ylabel('Frequency')
    ax5.set_title('Dark Band Timing\n(from estimated ages)')
    ax5.set_xticks(range(1, 13))
    ax5.set_xticklabels(month_names_short, rotation=45)
    ax5.legend()
    ax5.grid(alpha=0.3, axis='y')
else:
    ax5.text(0.5, 0.5, 'No dark bands', ha='center', va='center',
            transform=ax5.transAxes)

plt.tight_layout()
plt.savefig('corrected_age_model_results.png', dpi=150, bbox_inches='tight')
print("\n✓ Results figure saved")

# ============================================================================
# ADDITIONAL PLOT: Coral d18O vs Climate Synthetic d18O (calibration check)
# ============================================================================

fig2, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

# Panel 1: Coral d18O vs synthetic_d18o from climate
ax_a = axes[0]

# Coral d18O (observed)
ax_a.scatter(df_test['age_median'], df_test['d18o'],
            s=20, alpha=0.5, color='steelblue', label='Coral δ¹⁸O (observed)', zorder=3)

# Climate synthetic d18O (shifted to absolute scale)
d18o_offset_mean = float(np.mean(trace.posterior['d18o_offset'].values))
climate_overlap = df_climate[
    (df_climate['age_absolute'] >= df_test['age_median'].min()) &
    (df_climate['age_absolute'] <= df_test['age_median'].max())
]
ax_a.plot(climate_overlap['age_absolute'],
         climate_overlap['synthetic_d18o'] + d18o_offset_mean,
         color='red', linewidth=1.5, alpha=0.8,
         label=f'Climate predicted (offset={d18o_offset_mean:.2f}‰)', zorder=2)

ax_a.set_ylabel('δ¹⁸O (‰ VPDB)')
ax_a.set_title('Coral δ¹⁸O vs Climate Prediction\n(calibration check)',
              fontweight='bold')
ax_a.legend(fontsize=9)
ax_a.grid(alpha=0.3)
# Shade pre-climate period
ax_a.axvspan(df_test['age_median'].min(), 1958, alpha=0.1, color='gray',
            label='Pre-climate data')

# Panel 2: SST record
ax_b = axes[1]
ax_b.plot(climate_overlap['age_absolute'], climate_overlap['SST'],
         color='tomato', linewidth=1.2, label='SST (°C)')
ax_b.set_ylabel('SST (°C)')
ax_b.set_title('Sea Surface Temperature', fontweight='bold')
ax_b.legend(fontsize=9)
ax_b.grid(alpha=0.3)

# Panel 3: SSS record
ax_c = axes[2]
ax_c.plot(climate_overlap['age_absolute'], climate_overlap['SSS'],
         color='royalblue', linewidth=1.2, label='SSS (psu)')
ax_c.set_ylabel('SSS (psu)')
ax_c.set_xlabel('Calendar Year (CE)')
ax_c.set_title('Sea Surface Salinity', fontweight='bold')
ax_c.legend(fontsize=9)
ax_c.grid(alpha=0.3)

plt.suptitle('Climate Data Calibration Period (1985-2007)',
            fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('climate_calibration_check.png', dpi=150, bbox_inches='tight')
print("✓ Climate calibration figure saved")

# Save results
df_test.to_csv('age_estimates_corrected.csv', index=False)
print("✓ Age estimates saved to CSV")

# Summary statistics
summary = az.summary(trace, var_names=['mean_time_per_sample', 'age_I_start',
                                       'age_H_end', 'd18o_offset', 'd18o_noise'])
print("\nParameter estimates:")
print(summary)

print("\n" + "="*80)
print("RESULTS WITH CLIMATE CALIBRATION")
print("="*80)
print(f"\nAge estimates:")
print(f"  Span: {ages_median.min():.1f} to {ages_median.max():.1f} CE (~{ages_median.max() - ages_median.min():.1f} years)")
print(f"  Dark bands: {month_names[month_idx]} (circular mean: {mean_month_circ:.1f} ± {std_month_circ:.1f} months)")
print(f"  Mean uncertainty:   ±{df_test['age_uncertainty'].mean()/2:.2f} years")
print(f"  Youngest (2007):    ±{df_test['age_uncertainty'].iloc[-1]/2:.2f} years")
print(f"  Oldest (~1985):     ±{df_test['age_uncertainty'].iloc[0]/2:.2f} years")

d18o_offset_post = trace.posterior['d18o_offset'].values.flatten()
print(f"\nδ¹⁸O calibration:")
print(f"  Offset: {np.mean(d18o_offset_post):.3f} ± {np.std(d18o_offset_post):.3f} ‰")
print(f"  Full model: coral d18O = {np.mean(d18o_offset_post):.2f} + (-0.22*SST + 0.80*SSS)")

if not ages_increasing:
    print("\n⚠️  WARNING: Ages not strictly monotonic - constraint needs strengthening")
else:
    print("\n✅ Ages monotonically increasing")

print("\n✅ Next steps:")
print("1. Extend to all 8 paths")
print("2. Re-enable VonMises seasonal constraint")
print("3. Add δ¹³C seasonal/long-term decomposition (GP)")

In [ ]:
"""
Bayesian Age Model for Solomon Islands Coral
=============================================
Key design decisions based on data analysis:

1. XRAY COUNTS (age2 integers) are the PRIMARY chronology.
   Every integer value of age2 (60, 61, 62 ...) is a confirmed
   annual dark band = a known year marker.
   → These are treated as HARD anchors: age = collection_year - (age2_max - age2)
   → The model pins each integer age2 sample to its calendar year exactly.

2. DARK BAND SHAPE (age_end) tells us when the band ENDS each year.
   dark_start (April = 4/12) and dark_length (0.5 yr) from your code
   give us the seasonal timing within each year.
   → Used to constrain month-of-year for band start/end.

3. CLIMATE DATA (SST/SSS) is used for INTERANNUAL timing.
   synthetic_d18o = -0.22*SST + 0.80*SSS constrains the δ¹⁸O signal
   between the hard xray anchors — this is where climate helps.

4. Between anchors: interpolate with variable growth rate (Bayesian).
   The model only has freedom between integer age2 markers.

Structure of age2:
  Integer values (60, 61, ...) = dark band START = known calendar year
  age_end values (60.5, 61.5, ...) = dark band END = ~6 months later
  Non-integer, non-end values = interpolated between bands
"""

import pymc as pm
import numpy as np
import pandas as pd
import arviz as az
import matplotlib.pyplot as plt
import pytensor.tensor as pt
from scipy.interpolate import interp1d
from datetime import date

DATE = date.today().strftime('%Y%m%d')

# ── file paths ────────────────────────────────────────────────────────────────
CORAL_FILE   = '../../../2025-12-20_AgeModel-NewXrayPatch/age_model/lab_results.xlsx'
CLIMATE_FILE = '../../../2026-01-15_Climate-Model/analysis/climate_df.csv'

# ── load data ─────────────────────────────────────────────────────────────────
df         = pd.read_excel(CORAL_FILE)
df_climate = pd.read_csv(CLIMATE_FILE).sort_values('age_absolute').reset_index(drop=True)

print("=" * 80)
print("BAYESIAN CORAL AGE MODEL  —  XRAY COUNTS AS HARD ANCHORS")
print("=" * 80)

# ── subset to paths I and H ───────────────────────────────────────────────────
test_paths = ['I', 'H']
df_test    = df[df['path_name'].isin(test_paths)].copy().reset_index(drop=True)

collection_date = 2007.25   # April 2007, Sample H-113
n_samples       = len(df_test)   # 235

# ── seasonal parameters (from your code) ─────────────────────────────────────
dark_start  = 4/12 - 1    # April = month 4, expressed as fraction: -0.667
                           # meaning bands start in April of the PREVIOUS year?
                           # Actually: dark_start offsets age2 → age_lin
                           # age_lin = age2 + dark_start
                           # So band at age2=60 starts at age_lin=59.333 = April
dark_length = 0.5          # band lasts 5-6 months (April→September)

# The dark band START month in fractional year:
# age2=60 means the 60th band. With dark_start = -0.667:
# age_lin = 59.333 relative to oldest sample
# age_absolute = 1928.099 + 59.333 = 1987.432  (April 1987)
# This matches: 60th band starts April 1987
dark_start_frac = (4 / 12)   # April = 0.333 of year

print(f"Seasonal params: dark bands start ~April (month 4), last ~{dark_length*12:.0f} months")

# ============================================================================
# CORE INSIGHT: Build calendar ages from xray counts directly
# ============================================================================

print("\nBuilding xray-anchored age grid...")

# The youngest dark band in the full record is age2 = 79.0
# Collection date = 2007.25 (April 2007)
# The dark band at age2=79.0 should be at April 2007 - tiny bit before
# Actually from your data: age2=79.0 at path_H path_num=112, age_absolute=2007.049
# And age2=79.1 at path_num=113, age_absolute=2007.149 (last sample)

# Collection anchor: age2_max = 79.1 → 2007.149
# So calendar year = collection_date_of_oldest_sample + age_lin
# where age_lin = age2 + dark_start

# YOUR CONVERSION FORMULA (from your code):
#   age_lin     = age2 + dark_start          (for band starts)
#   age_lin_end = age_end + dark_start + dark_length   (for band ends)
#   age_absolute = anchor_year + age_lin
# where anchor_year is chosen so the youngest sample = collection_date

# Compute anchor_year from the last sample:
last_age2   = df_test['age2'].dropna().max()   # 79.1
anchor_year = collection_date - (last_age2 + dark_start)
print(f"  Anchor year (offset): {anchor_year:.4f}")
print(f"  Verification: last sample age = {anchor_year + last_age2 + dark_start:.4f} (should be {collection_date})")

# ── Build xray-derived age for every sample ──────────────────────────────────
# Step 1: assign age_lin for ALL rows using your interpolation logic
s1 = df_test['age2'].rename('age')   + dark_start
s2 = df_test['age_end'].rename('age') + dark_start + dark_length
df_test['age_lin_raw'] = s1.combine_first(s2)

# Step 2: interpolate within each path (your method)
from scipy.interpolate import interp1d as sp_interp1d

age_lin_interp = np.full(len(df_test), np.nan)
for p in test_paths:
    mask     = df_test['path_name'] == p
    mask_idx = df_test.index[mask]
    sub      = df_test[mask].copy()
    known    = sub[sub['age_lin_raw'].notna()]
    if len(known) >= 2:
        f   = sp_interp1d(known['path_num'], known['age_lin_raw'],
                          bounds_error=False, fill_value='extrapolate')
        age_lin_interp[mask_idx] = f(sub['path_num'].values)

df_test['age_lin']      = age_lin_interp
df_test['age_xray']     = anchor_year + df_test['age_lin']   # calendar age from xray

print(f"\nXray-derived age range: {df_test['age_xray'].min():.2f} to {df_test['age_xray'].max():.2f}")
print(f"(should span ~{collection_date - df_test['age_xray'].min():.1f} years)")

# ── Identify HARD ANCHOR samples (integer age2 = known year) ─────────────────
# These are samples where age2 is an integer → confirmed annual band
df_test['age2_is_integer'] = df_test['age2'].apply(
    lambda x: not pd.isna(x) and abs(x - round(x)) < 0.01
)
anchor_mask    = df_test['age2_is_integer']
anchor_indices = df_test.index[anchor_mask].values
anchor_ages    = df_test.loc[anchor_mask, 'age_xray'].values

print(f"\nHard anchors from integer age2: {len(anchor_indices)} samples")
print(f"  These are EXACT year markers from xray counting")

# Dark band start indices (age1 not null)
band_indices = df_test[df_test['age1'].notna()].index.values

print(f"Dark band starts: {len(band_indices)} samples")
print(f"  Path H bands: {(df_test.loc[band_indices,'path_name']=='H').sum()}")
print(f"  Path I bands: {(df_test.loc[band_indices,'path_name']=='I').sum()}")

# ── Climate interpolation (numpy, outside model) ──────────────────────────────
climate_start   = float(df_climate['age_absolute'].min())
climate_interp  = sp_interp1d(df_climate['age_absolute'].values,
                               df_climate['synthetic_d18o'].values,
                               kind='linear', bounds_error=False, fill_value=0.0)

# Evaluate at xray ages → gives climate prediction at each sample
synth_at_xray   = climate_interp(df_test['age_xray'].values).astype('float64')
in_calib_mask   = (df_test['age_xray'].values >= climate_start).astype('float64')

print(f"\nClimate calibration: {in_calib_mask.sum():.0f}/{n_samples} samples in 1958-2007 window")

# ── path geometry ─────────────────────────────────────────────────────────────
n_H      = int((df_test['path_name'] == 'H').sum())   # 113
n_I      = int((df_test['path_name'] == 'I').sum())   # 122
i_last_age2  = float(df_test.loc[df_test['path_name']=='I', 'age2'].max())
h_first_age2 = float(df_test.loc[df_test['path_name']=='H', 'age2'].min())
age2_gap     = i_last_age2 - h_first_age2   # +1.0

# ── samples per year per path (from xray counts) ──────────────────────────────
# Path H: 20 dark bands × exactly 1 year each → compute directly
h_bands_ages = df_test.loc[(df_test['path_name']=='H') & anchor_mask, 'age_xray'].values
h_span       = h_bands_ages.max() - h_bands_ages.min()
h_band_count = len(h_bands_ages) - 1
h_samps      = int((df_test['path_name']=='H').sum())

mean_inc_H = h_span / (h_samps - 1) if h_samps > 1 else 0.18
print(f"\nPath H: {h_samps} samples over {h_span:.1f} known years → "
      f"{mean_inc_H:.4f} yr/sample  ({1/mean_inc_H:.1f} samples/yr)")

i_bands_ages = df_test.loc[(df_test['path_name']=='I') & anchor_mask, 'age_xray'].values
i_span       = i_bands_ages.max() - i_bands_ages.min() if len(i_bands_ages) > 1 else 17.8
i_samps      = int((df_test['path_name']=='I').sum())
mean_inc_I   = i_span / (i_samps - 1) if i_samps > 1 else 0.15
print(f"Path I: {i_samps} samples over {i_span:.1f} known years → "
      f"{mean_inc_I:.4f} yr/sample  ({1/mean_inc_I:.1f} samples/yr)")

# ============================================================================
# BAYESIAN MODEL
# ============================================================================
#
# What the model does:
#
#  HARD CONSTRAINTS (from xray integer counts):
#    For each sample with integer age2, its age is FIXED to the xray value.
#    We enforce this via a very tight Normal(mu=age_xray, sigma=0.01).
#    sigma=0.01 = ~4 days — effectively a hard pin.
#
#  SOFT CONSTRAINTS (Bayesian):
#    - Time increments between anchors are Lognormal (variable growth)
#    - Annual spacing between consecutive dark bands ~ Normal(1.0, 0.10)
#    - Monotonicity: all age differences > 0
#
#  CLIMATE LIKELIHOOD:
#    coral d18O = d18o_offset + synthetic_d18o(age) + noise
#    Constrains the ages BETWEEN xray anchors using SST/SSS signal.
#    (Pre-1958: only d18o_offset, no climate)
#
# Key difference from previous version:
#   Previous: used weak prior on growth rate → model drifted 6 years
#   Now:      xray integer counts pin ages every ~1 year → drift impossible
#
print("\n" + "=" * 80)
print("BUILDING PYMC MODEL")
print("=" * 80)

# Pre-compute anchor ages as pytensor constants for use inside model
anchor_ages_pt = anchor_ages.astype('float64')   # numpy, used as observed

with pm.Model() as age_model:

    # ── growth rate priors ────────────────────────────────────────────────────
    # Separate mean increment for each path (they have different densities)
    mean_inc_H_rv = pm.Gamma('mean_inc_H', alpha=mean_inc_H*400, beta=400)
    mean_inc_I_rv = pm.Gamma('mean_inc_I', alpha=mean_inc_I*400, beta=400)
    # Sharp prior centered on xray-derived value — sampler can adjust ±10%
    
    variation = pm.HalfNormal('time_variation', sigma=0.02)
    # Very tight: growth rate barely varies within a path

    # ── PATH H: build backward from collection date ───────────────────────────
    last_age_H = pm.Normal('age_H_end', mu=collection_date, sigma=0.05)

    time_inc_H = pm.Lognormal('time_inc_H',
                               mu=pt.log(mean_inc_H_rv),
                               sigma=variation,
                               shape=n_H - 1)
    ages_H = pm.Deterministic(
        'ages_H',
        (last_age_H - pt.concatenate([[0], pt.cumsum(time_inc_H)]))[::-1]
    )

    # ── PATH I: build backward from its young end (constrained by H) ─────────
    last_age_I = pm.Normal('age_I_end',
                           mu=ages_H[0] + age2_gap,
                           sigma=0.3)

    time_inc_I = pm.Lognormal('time_inc_I',
                               mu=pt.log(mean_inc_I_rv),
                               sigma=variation,
                               shape=n_I - 1)
    ages_I = pm.Deterministic(
        'ages_I',
        (last_age_I - pt.concatenate([[0], pt.cumsum(time_inc_I)]))[::-1]
    )

    all_ages = pm.Deterministic('all_ages', pt.concatenate([ages_I, ages_H]))

    # ── HARD XRAY ANCHOR CONSTRAINTS ─────────────────────────────────────────
    # Pin every integer age2 sample to its known calendar year.
    # sigma=0.05 yr (~18 days) — tight but not numerically pathological.
    # This is the KEY fix: the model MUST match xray counts.
    ages_at_anchors = all_ages[anchor_indices]
    pm.Potential(
        'xray_anchors',
        pm.logp(
            pm.Normal.dist(mu=anchor_ages_pt, sigma=0.05),
            ages_at_anchors
        ).sum()
    )
    print(f"  ✓ xray anchor constraint: {len(anchor_indices)} hard pins (σ=0.05 yr)")

    # ── PATH ALIGNMENT ────────────────────────────────────────────────────────
    pm.Potential('path_alignment',
                pm.logp(pm.Normal.dist(mu=age2_gap, sigma=0.3),
                       last_age_I - ages_H[0]))

    # ── MONOTONICITY ──────────────────────────────────────────────────────────
    age_diffs = all_ages[1:] - all_ages[:-1]
    pm.Potential('monotonic',
                pm.logp(pm.Exponential.dist(lam=1.0 / 0.15),
                       pm.math.maximum(age_diffs, 0.001)).sum())

    # ── ANNUAL SPACING between dark bands ────────────────────────────────────
    if len(band_indices) > 1:
        band_ages_t  = all_ages[band_indices]
        band_spacings = band_ages_t[1:] - band_ages_t[:-1]
        pm.Potential('annual_spacing',
                    pm.logp(pm.Normal.dist(mu=1.0, sigma=0.10),
                           band_spacings).sum())
        # Tighter sigma=0.10 than before (was 0.15) because xray counts confirm annual

    # ── δ¹⁸O CLIMATE LIKELIHOOD ──────────────────────────────────────────────
    # Climate interpolation done OUTSIDE model in numpy (avoids searchsorted hang).
    # synth_at_xray is fixed numpy array evaluated at xray-derived ages.
    # As the model updates ages, this approximation is good because:
    #   - Xray anchors keep ages close to xray values
    #   - Climate varies slowly (monthly resolution)
    #   - Between anchors (~0.15 yr apart) climate changes little
    #
    # For a fully dynamic version: use pt.as_tensor + vectorized interp (already done).
    # But since anchors constrain ages tightly, the numpy precomputation is accurate.

    d18o_offset = pm.Normal('d18o_offset', mu=25.6, sigma=0.3)
    d18o_noise  = pm.HalfNormal('d18o_noise', sigma=0.2)

    # Expected d18O using precomputed climate values at xray ages
    synth_pt    = pt.as_tensor_variable(synth_at_xray)
    in_calib_pt = pt.as_tensor_variable(in_calib_mask)
    d18o_expected = d18o_offset + in_calib_pt * synth_pt

    pm.Potential('d18o_obs',
                pm.logp(pm.Normal.dist(mu=d18o_expected, sigma=d18o_noise),
                       df_test['d18o'].values.astype('float64')).sum())

    # ── δ¹³C (simplified — long-term trend is research focus, don't constrain) ─
    d13c_mean  = pm.Normal('d13c_mean', mu=-2.0, sigma=0.5)
    d13c_noise = pm.HalfNormal('d13c_noise', sigma=0.5)
    pm.Normal('d13c_obs', mu=d13c_mean, sigma=d13c_noise,
             observed=df_test['d13c'].values)

    print(f"  ✓ Model built: {len(age_model.free_RVs)} free variables")

# ============================================================================
# SAMPLE
# ============================================================================

print("\nStarting MCMC sampling...")

with age_model:
    trace = pm.sample(
        draws=1000,
        tune=1000,
        chains=2,
        cores=1,
        target_accept=0.95,
        random_seed=42,
        return_inferencedata=True,
        init='adapt_diag'
    )

print("✓ Sampling complete")

# ============================================================================
# EXTRACT RESULTS
# ============================================================================

def month_from_decimal_year(t):
    frac = t - np.floor(t)
    return (np.floor(frac * 12).astype(int) % 12) + 1

ages_I_post = trace.posterior['ages_I'].values.reshape(-1, n_I)
ages_H_post = trace.posterior['ages_H'].values.reshape(-1, n_H)
ages_all    = np.concatenate([ages_I_post, ages_H_post], axis=1)

ages_median = np.median(ages_all, axis=0)
ages_lower  = np.percentile(ages_all, 2.5,  axis=0)
ages_upper  = np.percentile(ages_all, 97.5, axis=0)

df_test['age_bayes']       = ages_median
df_test['age_bayes_lower'] = ages_lower
df_test['age_bayes_upper'] = ages_upper
df_test['age_uncertainty'] = ages_upper - ages_lower
# Include xray-derived age for direct comparison
# age_xray already computed above

# Compare against xray
diff = ages_median - df_test['age_xray'].values
print(f"\nResiduals (Bayes − xray):")
print(f"  Mean:  {diff.mean():.3f} yr")
print(f"  Std:   {diff.std():.3f} yr")
print(f"  Max:   {np.abs(diff).max():.3f} yr")
print(f"  >0.5yr: {(np.abs(diff)>0.5).sum()} samples")

ages_increasing = bool(np.all(np.diff(ages_median) > 0))
print(f"\nAges monotonically increasing: {ages_increasing}")

# Circular statistics for dark band timing
band_ages_med    = ages_median[band_indices]
band_months      = month_from_decimal_year(band_ages_med)
angles           = (band_months - 1) * 2 * np.pi / 12
sin_m, cos_m     = np.mean(np.sin(angles)), np.mean(np.cos(angles))
mean_angle       = np.arctan2(sin_m, cos_m)
mean_month_circ  = (mean_angle * 12 / (2 * np.pi)) % 12 + 1
R                = np.sqrt(sin_m**2 + cos_m**2)
std_month_circ   = np.sqrt(-2 * np.log(max(R, 1e-10))) * 12 / (2 * np.pi)
month_names      = ['Jan','Feb','Mar','Apr','May','Jun',
                    'Jul','Aug','Sep','Oct','Nov','Dec']
month_idx        = int(np.round(mean_month_circ)) - 1

print(f"\nDark band timing: {month_names[month_idx]} ± {std_month_circ:.1f} months")
print(f"  (your model assumed April = month 4)")

# ============================================================================
# SAVE RESULTS
# ============================================================================

df_test.to_csv(f'age_estimates_corrected_{DATE}.csv', index=False)
print(f"\n✓ Saved: age_estimates_corrected_{DATE}.csv")

# ============================================================================
# VISUALIZE
# ============================================================================

fig = plt.figure(figsize=(18, 11))
gs  = fig.add_gridspec(2, 3, hspace=0.35, wspace=0.3)

month_names_short = ['Jan','Feb','Mar','Apr','May','Jun',
                     'Jul','Aug','Sep','Oct','Nov','Dec']
colors = {'I': 'steelblue', 'H': 'darkorange'}

# ── Plot 1: Age model comparison ─────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, :2])
for p in test_paths:
    mask = df_test['path_name'] == p
    idx  = df_test.index[mask]
    ax1.fill_between(idx, ages_lower[mask], ages_upper[mask],
                     alpha=0.25, color=colors[p])
    ax1.plot(idx, ages_median[mask], color=colors[p],
             linewidth=1.5, label=f'Bayes path {p}')
    ax1.plot(idx, df_test.loc[mask, 'age_xray'], '--',
             color=colors[p], alpha=0.6, linewidth=1, label=f'Xray path {p}')

# Mark hard anchors
ax1.scatter(anchor_indices, anchor_ages,
            s=40, color='red', zorder=5, label='Xray anchors (integer age2)')
# Mark dark bands
ax1.scatter(band_indices, ages_median[band_indices],
            s=80, marker='|', color='black', linewidths=1.5, zorder=6,
            label='Dark band starts')

ax1.set_xlabel('Sample Index (Older → Younger)')
ax1.set_ylabel('Calendar Year (CE)')
ax1.set_title('Age Model: Bayesian (solid) vs Xray interpolation (dashed)\n'
             'Red dots = hard xray anchors | Ticks = dark bands', fontweight='bold')
ax1.legend(fontsize=8, ncol=3)
ax1.grid(alpha=0.3)

# ── Plot 2: Rose diagram ──────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 2], projection='polar')
if len(band_indices) > 0:
    counts, _ = np.histogram(band_months, bins=np.arange(0.5, 13.5, 1))
    theta      = np.linspace(0, 2*np.pi, 12, endpoint=False) + np.pi/2
    width      = 2 * np.pi / 12
    ax2.bar(theta, counts, width=width, color='steelblue',
           alpha=0.7, edgecolor='black', linewidth=1.2)
    mean_theta = mean_angle + np.pi/2
    mx = max(counts.max(), 1)
    ax2.arrow(mean_theta, 0, 0, mx * 0.8,
             head_width=0.3, head_length=mx * 0.15,
             fc='red', ec='red', linewidth=2.5, zorder=10)
    ax2.set_xticks(theta)
    ax2.set_xticklabels(month_names_short, fontsize=8)
    ax2.set_theta_direction(-1)
    ax2.set_theta_zero_location('N')
    ax2.set_title(f'Dark Band Timing\n{month_names[month_idx]} ± {std_month_circ:.1f} mo',
                 pad=20, fontweight='bold')
    ax2.grid(True, alpha=0.3)

# ── Plot 3: Residuals Bayes − xray ───────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 0])
for p in test_paths:
    mask = df_test['path_name'] == p
    ax3.scatter(df_test.loc[mask, 'age_xray'],
               diff[mask.values],
               s=15, alpha=0.6, color=colors[p], label=f'Path {p}')
ax3.axhline(0, color='red', linewidth=1.5, linestyle='--')
ax3.axhline( 0.25, color='gray', linewidth=0.8, linestyle=':')
ax3.axhline(-0.25, color='gray', linewidth=0.8, linestyle=':')
ax3.set_xlabel('Xray age (CE)')
ax3.set_ylabel('Residual: Bayes − Xray (yr)')
ax3.set_title(f'Residuals\nMean={diff.mean():.3f} yr, Std={diff.std():.3f} yr',
             fontweight='bold')
ax3.legend()
ax3.grid(alpha=0.3)

# ── Plot 4: d18O vs age ───────────────────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 1])
for p in test_paths:
    mask = df_test['path_name'] == p
    ax4.scatter(df_test.loc[mask, 'age_bayes'],
               df_test.loc[mask, 'd18o'],
               s=20, alpha=0.6, color=colors[p], label=f'Path {p}')
d18o_off_est = float(np.mean(trace.posterior['d18o_offset'].values))
ax4_twin = ax4.twinx()
clim_plot = df_climate[(df_climate['age_absolute'] >= ages_median.min()) &
                        (df_climate['age_absolute'] <= ages_median.max())]
ax4_twin.plot(clim_plot['age_absolute'],
             clim_plot['synthetic_d18o'] + d18o_off_est,
             color='red', linewidth=1, alpha=0.7, label='Climate predicted')
ax4.set_xlabel('Age (CE)')
ax4.set_ylabel('Coral δ¹⁸O (‰)')
ax4_twin.set_ylabel('Climate d18O (‰)', color='red')
ax4.set_title(f'δ¹⁸O vs Age\n(d18O offset = {d18o_off_est:.2f}‰)', fontweight='bold')
ax4.legend(loc='upper left', fontsize=8)
ax4.grid(alpha=0.3)

# ── Plot 5: Uncertainty profile ───────────────────────────────────────────────
ax5 = fig.add_subplot(gs[1, 2])
ax5.plot(ages_median, (ages_upper - ages_lower) / 2,
        color='darkred', linewidth=1.5, alpha=0.8)
ax5.axhline(0.25, color='gray', linestyle=':', label='±0.25 yr')
ax5.scatter(anchor_ages, np.zeros(len(anchor_ages)) + 0.01,
           s=30, color='red', zorder=5, label='Xray anchors')
ax5.set_xlabel('Age (CE)')
ax5.set_ylabel('Age uncertainty ± (yr, 95% CI half-width)')
ax5.set_title('Uncertainty Profile\n(should be near 0 at xray anchors)',
             fontweight='bold')
ax5.legend(fontsize=8)
ax5.grid(alpha=0.3)

plt.suptitle('Bayesian Coral Age Model  —  Xray Counts as Hard Anchors',
            fontsize=14, fontweight='bold')
plt.savefig(f'coral_age_model_{DATE}.png', dpi=150, bbox_inches='tight')
print(f"✓ Saved: coral_age_model_{DATE}.png")

# ── Climate calibration figure ────────────────────────────────────────────────
fig2, axes2 = plt.subplots(3, 1, figsize=(14, 11), sharex=True)
clim_range = df_climate[(df_climate['age_absolute'] >= ages_median.min()) &
                         (df_climate['age_absolute'] <= ages_median.max())]

axes2[0].scatter(df_test['age_bayes'], df_test['d18o'],
                s=15, alpha=0.5, color='steelblue', label='Coral δ¹⁸O')
axes2[0].plot(clim_range['age_absolute'],
             clim_range['synthetic_d18o'] + d18o_off_est,
             color='red', linewidth=1.2, label=f'Climate (offset={d18o_off_est:.2f}‰)')
axes2[0].axvspan(ages_median.min(), climate_start, alpha=0.08, color='gray',
                label='Pre-climate period')
axes2[0].set_ylabel('δ¹⁸O (‰)')
axes2[0].set_title('Coral δ¹⁸O vs Climate Prediction', fontweight='bold')
axes2[0].legend(fontsize=8)
axes2[0].grid(alpha=0.3)

axes2[1].plot(clim_range['age_absolute'], clim_range['SST'],
             color='tomato', linewidth=1.2)
axes2[1].set_ylabel('SST (°C)')
axes2[1].set_title('Sea Surface Temperature', fontweight='bold')
axes2[1].grid(alpha=0.3)

axes2[2].plot(clim_range['age_absolute'], clim_range['SSS'],
             color='royalblue', linewidth=1.2)
axes2[2].set_ylabel('SSS (psu)')
axes2[2].set_xlabel('Calendar Year (CE)')
axes2[2].set_title('Sea Surface Salinity', fontweight='bold')
axes2[2].grid(alpha=0.3)

plt.suptitle('Climate Calibration', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'climate_calibration_{DATE}.png', dpi=150, bbox_inches='tight')
print(f"✓ Saved: climate_calibration_{DATE}.png")

# ── Summary ───────────────────────────────────────────────────────────────────
print("\n" + "=" * 80)
print("SUMMARY")
print("=" * 80)
summary = az.summary(trace, var_names=['mean_inc_H', 'mean_inc_I',
                                       'age_H_end', 'age_I_end',
                                       'd18o_offset', 'd18o_noise'])
print(summary)
print(f"\nAge span: {ages_median.min():.2f} to {ages_median.max():.2f} CE")
print(f"Residuals vs xray: mean={diff.mean():.3f} yr, std={diff.std():.3f} yr, max={np.abs(diff).max():.3f} yr")
print(f"Monotonic: {ages_increasing}")
print(f"Dark band month: {month_names[month_idx]} ± {std_month_circ:.1f} mo")
d18o_off_post = trace.posterior['d18o_offset'].values.flatten()
print(f"d18O offset: {d18o_off_post.mean():.3f} ± {d18o_off_post.std():.3f} ‰")
print(f"\n✅ Next: extend to all 8 paths using same anchor strategy")